# HiSpoT *M. tuberculosis* Spoligotyping — **v8** (2026 update)
*Works with **HiSpoT amplicon** FASTQs **and** whole-genome (WGS) TB FASTQs. Matches the original Excel layout and integrates the built-in SITVIT dictionary.*

### What changed vs v7
1. **Two assay modes** via the new `assay` parameter:
   * `HiSpoT` – amplicon mode, **reproduces v7 results exactly** (same 8 legacy columns/values).
   * `WGS` – whole-genome mode with coverage-aware spacer calling.
   * `auto` – picks the mode per sample from the fraction of reads that carry a spacer (amplicon enriches the DR region; WGS does not).
2. **Faster scanning** – a single-pass Aho-Corasick scanner (with a dependency-free k-mer fallback) replaces the per-read 86× substring search. ~10× faster, which makes millions of WGS reads practical. Results are identical to the v7 substring method.
3. **Single-end support** and broader file-naming detection (`_R1_/_R2_`, `_R1./_R2.`, `_1./_2.`, `.R1./.R2.`, `.fastq/.fq`, gzipped or not).
4. **Robustness** – clear errors instead of bare `assert`s, tolerant FASTQ reading (won't crash on a truncated record), per-sample error isolation (one bad file no longer aborts the batch), case-insensitive matching, and a one-time **dictionary sanitiser** that removes malformed SITVIT entries.
5. **New QC columns** appended after the 8 legacy ones: `assay`, `n_present`, `est_depth`, `hit_frac`, `qc`.

Legacy column order is unchanged: `sample, binary, octal, SIT, family, total_reads, mean_per_spacer, cutoff`.


In [ ]:
# 1) Install helper libs
#   pyahocorasick gives the fast scanner; if it fails to install the notebook
#   automatically falls back to a pure-Python scanner, so this is best-effort.
!pip -q install tqdm openpyxl ipyfilechooser pyahocorasick


In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3) Pick FASTQ root folder
from ipyfilechooser import FileChooser
from IPython.display import display, HTML

fc = FileChooser('/content/drive/MyDrive')
fc.title = '<b>Select the folder that contains (or nests) your FASTQ files</b>'
fc.show_only_dirs = True
display(fc)
display(HTML('<em>After picking a folder run the next cell.</em>'))


FileChooser(path='/content/drive/MyDrive', filename='', title='<b>Select the folder that contains (or nests) y…

In [ ]:
# 4) Parameters
import os, glob, pandas as pd, datetime, re, gzip, json

#@title **Run parameters** { run: "auto" }
fastq_root        = ""        #@param {type:"string"}
operator_name     = "Israel"  #@param {type:"string"}

#@markdown ---
#@markdown **Assay / layout** (`auto` decides per sample)
assay             = "auto"    #@param ["auto", "HiSpoT", "WGS"]
layout            = "auto"    #@param ["auto", "paired", "single"]

#@markdown ---
#@markdown **HiSpoT (amplicon) calling — leave at defaults to reproduce v7**
threshold         = 0.15      #@param {type:"number"}
hispot_min_reads  = 10        #@param {type:"integer"}

#@markdown ---
#@markdown **WGS calling (coverage-aware)**
wgs_frac          = 0.10      #@param {type:"number"}
wgs_min_reads     = 5         #@param {type:"integer"}

#@markdown ---
#@markdown **Auto-detection & QC thresholds**
auto_enrich_frac   = 0.02     #@param {type:"number"}
hispot_qc_min_mean = 15.0     #@param {type:"number"}
wgs_qc_min_depth   = 8        #@param {type:"integer"}

# ---- resolve fastq_root (prefer the file chooser if the field is blank) ----
if not fastq_root:
    try:
        fastq_root = fc.selected_path
    except NameError:
        fastq_root = None

if not fastq_root or not os.path.isdir(fastq_root):
    raise ValueError(
        "fastq_root is not set or does not exist.\n"
        "  -> Select a folder with the chooser in cell 3, OR paste a path into the "
        "fastq_root field above, then re-run this cell."
    )

# ---- validate numeric parameters with clear messages ----
def _check(name, val, lo, hi):
    if not (lo <= val <= hi):
        raise ValueError(f"{name}={val} is out of range; expected {lo} <= {name} <= {hi}.")
_check("threshold",          threshold,          0.0, 1.0)
_check("wgs_frac",           wgs_frac,           0.0, 1.0)
_check("auto_enrich_frac",   auto_enrich_frac,   0.0, 1.0)
if hispot_min_reads < 1 or wgs_min_reads < 1:
    raise ValueError("hispot_min_reads and wgs_min_reads must be >= 1.")

# ---- timestamped output folder: HiSpoT_Fastq/{timestamp} ----
base_out   = "/content/drive/MyDrive/HiSpoT_Fastq"
timestamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_folder = os.path.join(base_out, timestamp)
os.makedirs(output_folder, exist_ok=True)

print(f"FASTQ root  : {fastq_root}")
print(f"Operator    : {operator_name}")
print(f"Assay       : {assay}   |  Layout: {layout}")
print(f"HiSpoT call : cutoff = max({hispot_min_reads}, {threshold} x mean_per_spacer)")
print(f"WGS call    : cutoff = max({wgs_min_reads}, {wgs_frac} x est_depth)")
print(f"Output dir  : {output_folder}")


FASTQ root  : /content/drive/MyDrive/NGS/מחקרים/HiSpoT paper/Paul/spo_ai_val_verWGS (1)
Operator    : Israel
Assay       : auto   |  Layout: auto
HiSpoT call : cutoff = max(10, 0.15 x mean_per_spacer)
WGS call    : cutoff = max(5, 0.1 x est_depth)
Output dir  : /content/drive/MyDrive/HiSpoT_Fastq/20260611_115140


In [ ]:
# 5) Discover FASTQ samples (paired and/or single-end, multiple naming styles)
_FASTQ_EXT = re.compile(r"\.(fastq|fq)(\.gz)?$", re.IGNORECASE)

def parse_mate(fname):
    """Return (stem, mate) for a FASTQ filename. mate is 1, 2 or None (single-end).
    Handles _R1_/_R2_ (Illumina, optional lane chunk), _R1/_R2, _1/_2, .R1/.R2 ."""
    base = _FASTQ_EXT.sub("", os.path.basename(fname))
    for rx in (r"_R([12])(?:_[0-9]+)?$", r"\.R([12])$", r"_([12])$"):
        m = re.search(rx, base, re.IGNORECASE)
        if m:
            return base[:m.start()], int(m.group(1))
    return base, None

# gather every fastq/fq(.gz) under the root (recursive)
all_fastqs = [p for p in glob.glob(os.path.join(fastq_root, "**", "*"), recursive=True)
              if _FASTQ_EXT.search(p)]
if not all_fastqs:
    raise FileNotFoundError(
        f"No .fastq/.fq(.gz) files found under {fastq_root} (searched all sub-folders)."
    )

mate1, mate2, singles = {}, {}, []
for p in sorted(all_fastqs):
    stem, mate = parse_mate(p)
    if   mate == 1: mate1[stem] = p
    elif mate == 2: mate2[stem] = p
    else:           singles.append((stem, p))

samples = []                      # list of (sample_name, [files...])
paired_keys = sorted(set(mate1) & set(mate2))
orphans = sorted(set(mate1) ^ set(mate2))

if layout in ("auto", "paired"):
    for k in paired_keys:
        samples.append((k, [mate1[k], mate2[k]]))

if layout == "single":
    # treat EVERY file as its own single-end sample
    samples = [(parse_mate(p)[0] or os.path.basename(p), [p]) for p in sorted(all_fastqs)]
elif layout == "auto":
    # add true single-end files; in auto, orphaned R1/R2 are also run single-end
    for stem, p in singles:
        samples.append((stem, [p]))
    for k in orphans:
        p = mate1.get(k) or mate2.get(k)
        samples.append((k, [p]))
elif layout == "paired" and orphans:
    print(f"⚠️  {len(orphans)} file(s) had no mate and were skipped (layout='paired'): "
          f"{orphans[:5]}{'...' if len(orphans) > 5 else ''}")

# de-duplicate by sample name (keep first), keep deterministic order
seen, deduped = set(), []
for name, files in samples:
    if name in seen:
        print(f"⚠️  Duplicate sample name '{name}' encountered; keeping the first occurrence.")
        continue
    seen.add(name); deduped.append((name, files))
samples = deduped

if not samples:
    raise FileNotFoundError(
        "No usable samples were assembled. Check file naming "
        "(_R1_/_R2_, _R1/_R2, _1/_2, .R1/.R2) and the layout setting."
    )

n_paired = sum(1 for _, f in samples if len(f) == 2)
n_single = sum(1 for _, f in samples if len(f) == 1)
print(f"Assembled {len(samples)} sample(s): {n_paired} paired, {n_single} single-end.")


Assembled 1 sample(s): 1 paired, 0 single-end.


In [ ]:
# 6) Spacer reference lists
fsp=["ATAGAGGGTCGCCGGCT", "CCTCATGCTTGGGCGAC", "CCGTGCTTCCAGTGATC", "ACGTCATACGCCGACCA", "TTTTCTGACCACTTGTG", "CGTCGTCATTTCCGGCT", "GAGGAGAGCGAGTACTC", "CGTGAAACCGCCCCCAG", "ACTCGGAATCCCATGTG", "TCGACACCCGCTCTAGT", "GTGAGCAACGGCGGCGG", "ATATCTGCTGCCCGCCC", "GACCATCATTGCCATTC", "GGTGTGATGCGGATGGT", "CTTGAATAACGCGCAGT", "CGAGTTCCCGTCAGCGT", "GCGCCGGCCCGCGCGGA", "CATGGACCCGGGCGAGC", "TAACTGGCTTGGCGCTG", "TTGACCTCGCCAGGAGA", "TCGATGTCGATGTCCCA", "ACCGCAGACGGCACGAT", "AGCATCGCTGATGCGGT", "CCGCCTGCTGGGTGAGA", "GATCAGCGACCACCGCA", "CTTCAGCACCACCATCA", "GGATTCGTGATCTCTTC", "TGCCCCGGCGTTTAGCG", "AAATACAGGCTCCACGA", "GGTTGCCCCGCGCCCTT", "TCAGACAGGTTCGCGTC", "GACCAAATAGGTATCGG", "GACATGACGGCGGTGCC", "AAGTCACCTCGCCCACA", "TCCGTACGCTCGAAACG", "CGAAATCCAGCACCACA", "CGCGAACTCGTCCACAG", "CGTGGATGGCGGATGCG", "GACGATGGCCAGTAAAT", "CGCCATCTGTGCCTCAT", "GGAGCTTTCCGGCTTCT", "ATGGTGGGACATGGACG", "CGCAGAATCGCACCGGG"]
crsp=["AGCCGGCGACCCTCTAT", "GTCGCCCAAGCATGAGG", "GATCACTGGAAGCACGG", "TGGTCGGCGTATGACGT", "CACAAGTGGTCAGAAAA", "AGCCGGAAATGACGACG", "GAGTACTCGCTCTCCTC", "CTGGGGGCGGTTTCACG", "CACATGGGATTCCGAGT", "ACTAGAGCGGGTGTCGA", "CCGCCGCCGTTGCTCAC", "GGGCGGGCAGCAGATAT", "GAATGGCAATGATGGTC", "ACCATCCGCATCACACC", "ACTGCGCGTTATTCAAG", "ACGCTGACGGGAACTCG", "TCCGCGCGGGCCGGCGC", "GCTCGCCCGGGTCCATG", "CAGCGCCAAGCCAGTTA", "TCTCCTGGCGAGGTCAA", "TGGGACATCGACATCGA", "ATCGTGCCGTCTGCGGT", "ACCGCATCAGCGATGCT", "TCTCACCCAGCAGGCGG", "TGCGGTGGTCGCTGATC", "TGATGGTGGTGCTGAAG", "GAAGAGATCACGAATCC", "CGCTAAACGCCGGGGCA", "TCGTGGAGCCTGTATTT", "AAGGGCGCGGGGCAACC", "GACGCGAACCTGTCTGA", "CCGATACCTATTTGGTC", "GGCACCGCCGTCATGTC", "TGTGGGCGAGGTGACTT", "CGTTTCGAGCGTACGGA", "TGTGGTGCTGGATTTCG", "CTGTGGACGAGTTCGCG", "CGCATCCGCCATCCACG", "ATTTACTGGCCATCGTC", "ATGAGGCACAGATGGCG", "AGAAGCCGGAAAGCTCC", "CGTCCATGTCCCACCAT", "CCCGGTGCGATTCTGCG"]

In [ ]:
# 7) SITVIT octal→(SIT,family) dictionary (trimmed to save space – full dict in original code)
sit_fam = {
    "000000000003771": ("1","Beijing"),
    "777777777760771": ("53","T1"),
    "703777740003771": ("485","CAS1-Delhi"),
              "000000004020771":["2","H2"],
        "000000007720771":["3","H3"],
        "000000007760771":["4","None"],
        "000677777760771":["5","T1"],
        "077777777413731":["6","EAI1-SOM"],
        "377777777760771":["7","T1"],
        "400037777413771":["8","EAI5"],
        "401776777760601":["9","X2"],
        "477777277413771":["10","EAI8-MDG"],
        "477777777413071":["11","EAI3-IND"],
        "500036777760771":["12","X3"],
        "500066777760771":["13","X3"],
        "500076777760771":["14","X3"],
        "577777600360771":["15","None"],
        "577777777413071":["16","EAI3-IND"],
        "677737607760771":["17","LAM2"],
        "677776777760601":["18","X2"],
        "677777477413771":["19","EAI2-Manila"],
        "677777607760771":["20","LAM1"],
        "703377400001771":["21","CAS1-Kili"],
        "703777400001771":["22","CAS1-Delhi"],
        "703777740001471":["23","CAS1-Delhi"],
        "703777740003031":["24","CAS1-Delhi"],
        "703777740003171":["25","CAS1-Delhi"],
        "703777740003771":["26","CAS1-Delhi"],
        "703777747770371":["27","None"],
        "737777777413371":["28","EAI5"],
        "760001400000171":["29","LAM"],
        "773776607760771":["30","LAM9"],
        "774037777760700":["31","T"],
        "776000000000171":["32","None"],
        "776177607760771":["33","LAM3"],
        "776377777760771":["34","S"],
        "777737777420771":["35","H4"],
        "777737777720771":["36","H3"],
        "777737777760771":["37","T3"],
        "777776407760601":["38","X2"],
        "777777347760471":["39","T4-CEU1"],
        "777777377760771":["40","T4"],
        "777777404760771":["41","LAM7-TUR"],
        "777777607760771":["42","LAM9"],
        "777777747413771":["43","EAI6-BGD1"],
        "777777757760771":["44","T5"],
        "777777764020771":["45","H1"],
        "777777770000000":["46","None"],
        "777777774020771":["47","H1"],
        "777777777413731":["48","EAI1-SOM"],
        "777777777720731":["49","H3"],
        "777777777720771":["50","H3"],
        "777777777760700":["51","T1"],
        "777777777760731":["52","T2"],
        "777777777760771":["53","T1"],
        "777777777763771":["54","Manu2"],
        "776377775540771":["55","S"],
        "777737770000000":["56","None"],
        "777777143760771":["57","LAM10-CAM"],
        "777777557760771":["58","T5-Madrid2"],
        "777777606060771":["59","LAM11-ZWE"],
        "777777607760731":["60","LAM4"],
        "777777743760771":["61","LAM10-CAM"],
        "777777774020731":["62","H1"],
        "377777737760771":["63","T1"],
        "777777607560771":["64","LAM6"],
        "777777777760471":["65","T1"],
        "777777037720671":["66","H3"],
        "777777037720771":["67","H3"],
        "775777757760771":["68","T5"],
        "SUPPRESSED":["69","None"],
        "700076777760671":["70","X3"],
        "776337777760771":["71","S"],
        "777577777003731":["72","EAI1-SOM"],
        "777737777760731":["73","Ambiguous:T3 T2"],
        "777777600160771":["74","None"],
        "777767777720771":["75","H3"],
        "474001377413771":["76","EAI8-MDG"],
        "743777777760771":["77","T1"],
        "777777777760711":["78","T"],
        "517347606060661":["79","LAM11-ZWE"],
        "767777744020771":["80","H1"],
        "777377607760771":["81","LAM9"],
        "777757606060731":["82","LAM11-ZWE"],
        "777757777760671":["83","T1"],
        "777773604060731":["84","LAM4"],
        "557757606060771":["85","LAM11-ZWE"],
        "777777737760771":["86","T1"],
        "577757606060771":["87","LAM11-ZWE"],
        "776337777760731":["88","S"],
        "674000003413771":["89","EAI2-Nonthaburi"],
        "677777776000371":["90","None"],
        "700036777760771":["91","X3"],
        "700076777760771":["92","X3"],
        "777737607760771":["93","LAM5"],
        "777777347560471":["94","T4-CEU1"],
        "777777607560731":["95","LAM6"],
        "777777750003371":["96","EAI7-BGD2"],
        "777777760760771":["97","T1"],
        "170003607760771":["98","LAM"],
        "757777777720771":["99","H3"],
        "777777777773771":["100","Manu1"],
        "774077606767071":["101","AFRI_2"],
        "777703777760771":["102","T"],
        "777777754020731":["103","H1"],
        "576167400000171":["104","None"],
        "776160000000071":["105","None"],
        "776177400000171":["106","None"],
        "036377777760771":["107","S"],
        "377777777720731":["108","H3"],
        "477777377413771":["109","EAI8-MDG"],
        "770777775720771":["110","H3"],
        "776167607760771":["111","LAM3"],
        "777737761760771":["112","T3"],
        "477777776413071":["113","EAI3-IND"],
        "777767741760771":["114","T1"],
        "777767743760771":["115","LAM10-CAM"],
        "777767775720771":["116","H3"],
        "777767777760731":["117","T2"],
        "777767777760771":["118","T1"],
        "777776777760771":["119","X1"],
        "777777577760771":["120","T1"],
        "777777775720771":["121","H3"],
        "777777775760771":["122","T1"],
        "777777776360771":["123","T1"],
        "777777777700771":["124","None"],
        "000000007760731":["125","LAM3"],
        "477777777413771":["126","EAI5"],
        "577777777420771":["127","H4"],
        "637775777760730":["128","T2-Uganda"],
        "700777747413771":["129","EAI6-BGD1"],
        "776177607760731":["130","LAM3"],
        "777717777760771":["131","T1"],
        "777777606000031":["132","None"],
        "777603405760430":["133","Ambiguous:LAM5 LAM4"],
        "777777777720631":["134","H3"],
        "777777777760730":["135","T2-Uganda"],
        "777603405760471":["136","LAM5"],
        "777776777760601":["137","X2"],
        "777777777413700":["138","EAI5"],
        "777777774413771":["139","EAI4-VNM"],
        "702777740003171":["140","CAS1-Delhi"],
        "703767740003771":["141","CAS1-Delhi"],
        "703777700003771":["142","CAS1-Delhi"],
        "757400004020771":["143","H1"],
        "770000003760771":["144","T1"],
        "770007000003071":["145","AFRI"],
        "774000077777031":["146","None"],
        "774777616000371":["147","None"],
        "774777716000371":["148","None"],
        "777000377760771":["149","T3-ETH"],
        "777767607760771":["150","LAM9"],
        "777767774020771":["151","H1"],
        "777777776413771":["152","EAI5"],
        "757777777760731":["153","T2"],
        "757777777760771":["154","T1"],
        "776157777760771":["155","T1"],
        "776177777760771":["156","None"],
        "777737777760471":["157","T3"],
        "777737777760601":["158","T3"],
        "777740017760771":["159","None"],
        "777777477400001":["160","None"],
        "777777607740771":["161","LAM9"],
        "777777607760671":["162","LAM9"],
        "777777607760700":["163","LAM4"],
        "777777760060771":["164","T1"],
        "777777774720731":["165","H3"],
        "777777777320571":["166","H3"],
        "777777777660771":["167","T1"],
        "777777777720671":["168","H3"],
        "777777777740700":["169","T1"],
        "776017777760400":["170","T1"],
        "776017777760771":["171","T1"],
        "777777777740771":["172","T1"],
        "777767677760771":["173","T1"],
        "777777037760771":["174","None"],
        "777777677760731":["175","T2"],
        "777737607560771":["176","LAM6"],
        "377777607760771":["177","LAM9"],
        "477737607760731":["178","LAM2"],
        "677737607760711":["179","LAM2"],
        "677777777720771":["180","H3"],
        "770777777777671":["181","AFRI_1"],
        "777777004020771":["182","H1"],
        "777777377720771":["183","H3"],
        "777417606060731":["184","LAM11-ZWE"],
        "777766777760601":["185","X2"],
        "777767404760771":["186","LAM7-TUR"],
        "770777777767671":["187","AFRI_1"],
        "770777777777631":["188","AFRI_1"],
        "777741777760771":["189","T1"],
        "000000000003731":["190","Beijing"],
        "177777777760771":["191","T1"],
        "400003777413771":["192","EAI5"],
        "677737607660771":["193","LAM2"],
        "677737607760731":["194","LAM2"],
        "677777607660771":["195","LAM1"],
        "677777777760771":["196","T1"],
        "700036777760731":["197","X3"],
        "700040777760771":["198","T1"],
        "700076760000011":["199","X"],
        "700076777760700":["200","X3"],
        "700377000003771":["201","CAS"],
        "701776777760601":["202","X2"],
        "703777740001771":["203","CAS"],
        "737777777413771":["204","EAI5"],
        "737777777760771":["205","T1"],
        "740777607760771":["206","LAM9"],
        "767777777720771":["207","H3"],
        "774077774413771":["208","EAI4-VNM"],
        "776017607760771":["209","LAM12-Madrid1"],
        "776037777760771":["210","T1"],
        "776137607760771":["211","LAM3"],
        "776377776760771":["212","S"],
        "776777607760731":["213","LAM4"],
        "777703757760771":["214","T5"],
        "777703777360771":["215","T"],
        "777717607760771":["216","LAM5"],
        "777736777760771":["217","X1"],
        "777737774020771":["218","H1"],
        "777740777760771":["219","T1"],
        "777742777760601":["220","X2"],
        "777766777760771":["221","X1"],
        "777774077560771":["222","None"],
        "777776377760601":["223","T1"],
        "777776776360711":["224","X1"],
        "777776777760611":["225","X1"],
        "777776777763771":["226","Manu2"],
        "777777370020771":["227","H3"],
        "777777603560771":["228","T1"],
        "777777603760371":["229","T1"],
        "777777603760771":["230","LAM9"],
        "777777757760700":["231","T5"],
        "777777760000171":["232","None"],
        "777777770060731":["233","T2"],
        "777777777413371":["234","EAI5"],
        "777777777413531":["235","EAI1-SOM"],
        "777777777413771":["236","EAI5"],
        "777777777700000":["237","None"],
        "777777777700171":["238","None"],
        "777777777760031":["239","T2"],
        "777777777760371":["240","T"],
        "777777777760411":["241","T1"],
        "777777777760431":["242","T1"],
        "777777777760600":["243","T1"],
        "777777777760601":["244","T1"],
        "777777777760671":["245","T1"],
        "777777777777731":["246","None"],
        "703777740003471":["247","CAS1-Delhi"],
        "763777774413771":["248","EAI4-VNM"],
        "777640007760771":["249","LAM"],
        "000000000000371":["250","Beijing"],
        "774000000760771":["251","T1"],
        "777477607760771":["252","LAM9"],
        "777777663760771":["253","T1"],
        "777760007760771":["254","T5-RUS1"],
        "000000000003671":["255","Beijing"],
        "777777777413671":["256","EAI5"],
        "077777777413631":["257","EAI5"],
        "000000177760771":["258","T"],
        "775777377760771":["259","T4"],
        "000000000003171":["260","Beijing"],
        "737777773760771":["261","T1"],
        "774777777420771":["262","H3"],
        "777600003760771":["263","None"],
        "777740003760771":["264","T5-RUS1"],
        "000000000003371":["265","Beijing"],
        "775740003760771":["266","T1"],
        "777700003760771":["267","None"],
        "777777677720771":["268","H3"],
        "000000000000771":["269","Beijing"],
        "337777777413731":["270","EAI1-SOM"],
        "776000000000111":["271","None"],
        "037777777760771":["272","T1"],
        "777617607760771":["273","LAM5"],
        "777777600000371":["274","None"],
        "777777600020771":["275","H3"],
        "777777770060771":["276","T1"],
        "777777777560700":["277","T1"],
        "777777777760761":["278","T1"],
        "677677477413771":["279","EAI2-Manila"],
        "770000777760771":["280","T1-RUS2"],
        "777775777760771":["281","T1"],
        "777776747760601":["282","X2"],
        "777777704020771":["283","H1"],
        "037637777760771":["284","T1"],
        "637777777760771":["285","T1"],
        "677777470000000":["286","EAI2-Manila"],
        "677777477413751":["287","EAI2-Manila"],
        "700377740003771":["288","CAS2"],
        "703777740003571":["289","CAS1-Delhi"],
        "777777606760771":["290","LAM8"],
        "777777677760771":["291","T1"],
        "777777757413371":["292","EAI6-BGD1"],
        "777777777720751":["293","H3"],
        "577777777720771":["294","H3"],
        "776377763760771":["295","S"],
        "477567777413071":["296","EAI3-IND"],
        "477767377413671":["297","EAI8-MDG"],
        "477767777413071":["298","EAI3-IND"],
        "477776077411171":["299","EAI3-IND"],
        "577727674413671":["300","EAI4-VNM"],
        "577767777413631":["301","EAI5"],
        "777756777760771":["302","X1"],
        "777767777413331":["303","EAI1-SOM"],
        "540777757760771":["304","T5"],
        "777777477760631":["305","T-H37Rv_var4"],
        "777777601560771":["306","T1"],
        "077777777720731":["307","H3"],
        "674001777677600":["308","BOV"],
        "703767740003171":["309","CAS1-Delhi"],
        "777727770020720":["310","H3"],
        "777767570020731":["311","H3"],
        "777767770020731":["312","H3"],
        "777774003760731":["313","T2"],
        "777777570020731":["314","H3"],
        "777777764020731":["315","H1"],
        "777777770020731":["316","H3"],
        "777777777360731":["317","T2"],
        "570777777777671":["318","AFRI_1"],
        "574077607777071":["319","AFRI_2"],
        "770003606377071":["320","AFRI_2"],
        "770737004177671":["321","AFRI_1"],
        "770767777737671":["322","AFRI_1"],
        "770767777767671":["323","AFRI_1"],
        "770767777777671":["324","AFRI_1"],
        "770777507777671":["325","AFRI_1"],
        "770777707777671":["326","AFRI_1"],
        "774067606727071":["327","AFRI_2"],
        "774067606767071":["328","AFRI_2"],
        "774067607777071":["329","AFRI_2"],
        "774077607777031":["330","AFRI_2"],
        "774077607777071":["331","AFRI_2"],
        "776707777737071":["332","None"],
        "777727607760771":["333","LAM5"],
        "577777777760771":["334","T1"],
        "777777607720771":["335","None"],
        "777776777760731":["336","Ambiguous:T2 X1"],
        "777777777013731":["337","EAI1-SOM"],
        "077777777413071":["338","EAI3-IND"],
        "437776777760601":["339","X2"],
        "474377777413771":["340","EAI5"],
        "477777775413031":["341","EAI5"],
        "677777777413771":["342","EAI5"],
        "700000007175771":["343","None"],
        "700077777760771":["344","T1"],
        "777000377760731":["345","T3-ETH"],
        "777377700003531":["346","None"],
        "777736777760601":["347","X2"],
        "777736777760671":["348","Ambiguous:T3 X1"],
        "777737777413731":["349","EAI1-SOM"],
        "777776617760601":["350","X2"],
        "777776777600171":["351","None"],
        "777776777760711":["352","X1"],
        "777777774760771":["353","T1"],
        "777777777403731":["354","None"],
        "477777777413031":["355","EAI3-IND"],
        "703777600001771":["356","CAS1-Delhi"],
        "703777740000771":["357","CAS1-Delhi"],
        "717777777760771":["358","T1"],
        "037677774020771":["359","H1"],
        "540377777720771":["360","H3"],
        "577737777420771":["361","H4"],
        "675777757760771":["362","T5"],
        "716377777760771":["363","S"],
        "757777776760771":["364","T1"],
        "777000003760771":["365","T1"],
        "777577777760471":["366","T1"],
        "777737404760771":["367","LAM7-TUR"],
        "777777007720731":["368","H3"],
        "777777574020771":["369","H1"],
        "777777747760471":["370","T1"],
        "777777007720771":["371","H3"],
        "777777752000000":["372","None"],
        "777777767760771":["373","T1"],
        "777777776000771":["374","None"],
        "077737607760771":["375","LAM2"],
        "376177607760771":["376","LAM3"],
        "657737607760771":["377","LAM2"],
        "777777667760771":["378","T1"],
        "777741017760771":["379","None"],
        "577777777413771":["380","EAI5"],
        "703777740003071":["381","CAS1-Delhi"],
        "757777774020771":["382","H1"],
        "776377774020771":["383","H1"],
        "777777674020771":["384","H1"],
        "777736377760571":["385","T3"],
        "000000007360771":["386","LAM3"],
        "400017600560731":["387","T2"],
        "737777607760771":["388","LAM9"],
        "677767607760771":["389","LAM1"],
        "777777777620771":["390","H3"],
        "743777607760731":["391","LAM4"],
        "777777600160731":["392","T2"],
        "777757777760771":["393","T1"],
        "777777777620731":["394","H3"],
        "777777777760400":["395","T1"],
        "777777607560371":["396","LAM6"],
        "777777600000771":["397","LAM"],
        "777777607760631":["398","LAM4"],
        "777777777420410":["399","H3"],
        "757777774020731":["400","H1"],
        "776377403760771":["401","S"],
        "777777600000000":["402","None"],
        "777777743760731":["403","LAM10-CAM"],
        "777647000000011":["404","ZERO"],
        "777777000000011":["405","ZERO"],
        "000000000000731":["406","Beijing"],
        "000036040003771":["407","None"],
        "157756777760771":["408","X1"],
        "264073777777600":["409","BOV_2"],
        "377774077760771":["410","T1"],
        "407777777760771":["411","T1"],
        "437777606060600":["412","LAM11-ZWE"],
        "477777477413771":["413","EAI5"],
        "477777767413071":["414","EAI3-IND"],
        "477777774137771":["415","None"],
        "616777677777600":["416","BOV"],
        "577777770060771":["417","T1"],
        "617777777720771":["418","H3"],
        "630000207760771":["419","LAM3"],
        "637774777760730":["420","T2-Uganda"],
        "676600000000371":["421","None"],
        "676773667777600":["422","BOV_1"],
        "677774207760771":["423","LAM1"],
        "677777477413711":["424","EAI2-Manila"],
        "700076774160771":["425","X3"],
        "700377600003771":["426","CAS"],
        "703707740003771":["427","CAS1-Delhi"],
        "703777740003371":["428","CAS1-Delhi"],
        "703777740003731":["429","CAS1-Delhi"],
        "704003347760471":["430","T4-CEU1"],
        "717777774020771":["431","H1"],
        "735777777760731":["432","T2"],
        "757777777320771":["433","H3"],
        "760777777740371":["434","T1"],
        "763777607760771":["435","LAM9"],
        "770376777760601":["436","X2"],
        "774077777775071":["437","AFRI_3"],
        "774077777777071":["438","AFRI_3"],
        "777376777760601":["439","X2"],
        "777607607760771":["440","LAM5"],
        "777700277760771":["441","T1"],
        "777737377760771":["442","Ambiguous:T4 T3"],
        "777737607700771":["443","None"],
        "777740001760771":["444","T1"],
        "777756777760601":["445","X2"],
        "777760016760601":["446","T-Tuscany"],
        "777760377760771":["447","T1"],
        "777767777720731":["448","H3"],
        "777776757760771":["449","Ambiguous:X1 T5"],
        "777776770000000":["450","None"],
        "777777477760771":["451","T-H37Rv"],
        "777777606360771":["452","LAM9"],
        "777777677560771":["453","T1"],
        "777777736760731":["454","T2"],
        "777777760360771":["455","T1"],
        "777777774413711":["456","EAI4-VNM"],
        "777777777320771":["457","H3"],
        "777777777403771":["458","EAI5"],
        "777777777410331":["459","EAI1-SOM"],
        "777777777413751":["460","EAI5"],
        "777777777416761":["461","None"],
        "777777777560771":["462","T1"],
        "777777777720571":["463","H3"],
        "777777777747771":["464","None"],
        "777417737700000":["465","None"],
        "776377377760771":["466","S"],
        "000000000020771":["467","H"],
        "067777777413031":["468","EAI5"],
        "077777607760771":["469","LAM1"],
        "177777777413771":["470","EAI5"],
        "303777740003771":["471","CAS1-Delhi"],
        "377777777720771":["472","H3"],
        "401777777413071":["473","EAI3-IND"],
        "467777777413031":["474","EAI5"],
        "477776077411771":["475","EAI5"],
        "577776777760601":["476","X2"],
        "577777777413731":["477","EAI1-SOM"],
        "617776777760601":["478","X2"],
        "640013777777600":["479","BOV_3"],
        "674777477411771":["480","EAI2-Manila"],
        "676773677777600":["481","BOV_1"],
        "676773777777600":["482","BOV_1"],
        "677777477413701":["483","EAI2-Manila"],
        "700036777740171":["484","X3"],
        "703777400003771":["485","CAS1-Delhi"],
        "703777740000371":["486","CAS"],
        "740000004020771":["487","H1"],
        "760777774020771":["488","H1"],
        "767776777760601":["489","X2"],
        "767776777760771":["490","X1"],
        "773776777760601":["491","X2"],
        "773777607760771":["492","LAM9"],
        "774177777413731":["493","EAI1-SOM"],
        "776377777760711":["494","S"],
        "776760003760771":["495","T5-RUS1"],
        "777460007760771":["496","T5-RUS1"],
        "777677777413771":["497","EAI5"],
        "777677777760771":["498","T1"],
        "777700000060771":["499","None"],
        "777700001760771":["500","None"],
        "777700377760771":["501","T1"],
        "777727737760771":["502","T3"],
        "777737377760731":["503","Ambiguous:T4 T3 T2"],
        "777737737760771":["504","T3"],
        "777737777760700":["505","T3"],
        "777760003760771":["506","T5-RUS1"],
        "777776037760771":["507","T1"],
        "777776377760771":["508","T1"],
        "777777605760771":["509","LAM9"],
        "777777700003531":["510","None"],
        "777777700020771":["511","H3"],
        "777777707720771":["512","H3"],
        "777777717760731":["513","T2"],
        "777777774413731":["514","EAI4-VNM"],
        "777777776760731":["515","T2"],
        "777777777360771":["516","T1"],
        "777777777413711":["517","EAI5"],
        "777777777660031":["518","T1"],
        "777777777740371":["519","T1"],
        "777777777760571":["520","T1"],
        "777777777760611":["521","T1"],
        "777777777760770":["522","T1"],
        "777777777777771":["523","Manu_ancestor"],
        "777777777720711":["524","H3"],
        "570777637743671":["525","AFRI_1"],
        "677777777403671":["526","None"],
        "677777777403771":["527","EAI5"],
        "677777777413671":["528","EAI5"],
        "677777777413731":["529","EAI1-SOM"],
        "770777477775671":["530","AFRI_1"],
        "737777774020771":["531","H1"],
        "770777767777671":["532","AFRI_1"],
        "777400777720771":["533","H3"],
        "777777607400000":["534","None"],
        "777777707760771":["535","T1"],
        "700000377776671":["536","AFRI_1"],
        "770777777777601":["537","AFRI_1"],
        "777637777413731":["538","EAI1-SOM"],
        "000000000000600":["539","Microti"],
        "000000000003031":["540","Beijing"],
        "000000000003711":["541","Beijing"],
        "077757607760731":["542","Ambiguous:LAM1 LAM4"],
        "577777677561771":["543","None"],
        "600000000760731":["544","T2"],
        "675737607760771":["545","LAM2"],
        "700036777560771":["546","X3"],
        "700036777760031":["547","X3"],
        "700076700000071":["548","CAS"],
        "700076757760771":["549","X3"],
        "717777477413731":["550","EAI1-SOM"],
        "737777777760610":["551","T1"],
        "740076777760601":["552","X2"],
        "741737777760731":["553","T3-OSA"],
        "757700777760771":["554","T1"],
        "757776777760571":["555","X1"],
        "773603774360771":["556","T1"],
        "773777774020771":["557","H1"],
        "776377770760731":["558","S"],
        "776377770760771":["559","S"],
        "777000000000371":["560","None"],
        "777500003760771":["561","T1"],
        "777737600000000":["562","None"],
        "777737770000071":["563","None"],
        "777737774413771":["564","EAI4-VNM"],
        "777737777760671":["565","T3"],
        "777740777760731":["566","T2"],
        "777760007763771":["567","Manu2"],
        "777767477760771":["568","T-H37Rv_var1"],
        "777767760000000":["569","None"],
        "777773776760771":["570","T1"],
        "777775774020771":["571","H1"],
        "777776407360601":["572","X2"],
        "777777663560771":["573","T"],
        "777777777740071":["574","T1"],
        "777777777760561":["575","T1"],
        "777776777360771":["576","X1"],
        "477767277413771":["577","EAI8-MDG"],
        "637777607760771":["578","LAM1"],
        "777737607700171":["579","None"],
        "676077607760771":["580","LAM3"],
        "777777677560671":["581","T1"],
        "777002777760731":["582","Ambiguous:T2 X1"],
        "777737777763771":["583","Manu2"],
        "777775777760731":["584","T2"],
        "000000000000031":["585","Beijing"],
        "000000007720631":["586","H3"],
        "700740007774671":["587","None"],
        "703777744003771":["588","None"],
        "776161000000071":["589","None"],
        "777777403760730":["590","T2-Uganda"],
        "777777757413771":["591","EAI6-BGD1"],
        "000000000101000":["592","Canettii"],
        "074000037777600":["593","PINI1"],
        "656573777777600":["594","BOV_1"],
        "676777777777600":["595","BOV"],
        "457777777420771":["596","H4"],
        "567777777420771":["597","H4"],
        "703777400000371":["598","CAS"],
        "703777400000771":["599","CAS"],
        "703777400003171":["600","CAS1-Delhi"],
        "700377400000771":["601","CAS"],
        "777777770000771":["602","None"],
        "777777777320731":["603","H3"],
        "677737607760751":["604","LAM2"],
        "777737607700031":["605","None"],
        "777737607760000":["606","Ambiguous:LAM5 LAM4"],
        "777777467400000":["607","None"],
        "776177770000000":["608","None"],
        "777600774020771":["609","H1"],
        "777767774020731":["610","H1"],
        "777767777560771":["611","T1"],
        "777777777760751":["612","T1"],
        "777177777760771":["613","T1"],
        "777777607760431":["614","LAM4"],
        "777777777720770":["615","H3"],
        "400000000003771":["616","None"],
        "777737776413771":["617","EAI5"],
        "777737777413771":["618","EAI5"],
        "777777002000011":["619","None"],
        "777777744020771":["620","H1"],
        "000000000002771":["621","Beijing"],
        "777601774413771":["622","EAI4-VNM"],
        "777777757777771":["623","None"],
        "477775777413071":["624","EAI3-IND"],
        "477777777412071":["625","EAI5"],
        "477777777412771":["626","EAI5"],
        "741737777760771":["627","T3-OSA"],
        "777777777760760":["628","T1"],
        "477777777410071":["629","EAI5"],
        "771777737760771":["630","T1"],
        "777777767720771":["631","H3"],
        "000000000003571":["632","Beijing"],
        "657777607760731":["633","Ambiguous:LAM1 LAM4"],
        "000000004177600":["634","PINI2"],
        "000000007560771":["635","LAM3"],
        "000000007777000":["636","PINI2"],
        "000000007777600":["637","PINI2"],
        "001777607760771":["638","LAM1"],
        "040000000000600":["639","PINI"],
        "074000007177200":["640","PINI"],
        "074000030000600":["641","PINI1"],
        "074000032000600":["642","PINI1"],
        "077777377000371":["643","None"],
        "200003377207600":["644","BOV_4-Caprae"],
        "200003717207600":["645","BOV_4-Caprae"],
        "200003777207600":["646","BOV_4-Caprae"],
        "200003777377600":["647","BOV_4-Caprae"],
        "200103717207600":["648","BOV_4-Caprae"],
        "377777423560771":["649","T1"],
        "436773677774600":["650","BOV"],
        "464063777777600":["651","BOV"],
        "477577777413071":["652","EAI3-IND"],
        "477777604020771":["653","H1"],
        "477777737413071":["654","EAI3-IND"],
        "477777777720771":["655","H3"],
        "577777377420771":["656","H4"],
        "600000016177600":["657","BOV"],
        "600000017777600":["658","BOV"],
        "602733777777600":["659","BOV_1"],
        "602773607777600":["660","BOV_1"],
        "602773777777600":["661","BOV_1"],
        "614073777777600":["662","BOV"],
        "616773677777600":["663","BOV_1"],
        "616773776377600":["664","BOV_1"],
        "616773777777600":["665","BOV_1"],
        "640013777177600":["666","BOV_3"],
        "644073777777600":["667","BOV"],
        "656473737777600":["668","BOV_1"],
        "656573377777200":["669","BOV_1"],
        "656573377777600":["670","BOV_1"],
        "664043777777600":["671","BOV"],
        "664063777777600":["672","BOV"],
        "664070377777600":["673","BOV"],
        "664071777777600":["674","BOV"],
        "664073377777600":["675","BOV_2"],
        "664073774037600":["676","BOV_2"],
        "664073777740600":["677","BOV_2"],
        "664073777757600":["678","BOV_2"],
        "664073777763600":["679","BOV_2"],
        "664073777767600":["680","BOV_2"],
        "664073777776600":["681","BOV_2"],
        "664073777777200":["682","BOV_2"],
        "664073777777600":["683","BOV_2"],
        "666773677777600":["684","BOV_1"],
        "674073777777600":["685","BOV"],
        "674153777777600":["686","BOV"],
        "674573777767600":["687","BOV"],
        "676473777777600":["688","BOV_1"],
        "676573777017600":["689","BOV_1"],
        "676573777637600":["690","BOV_1"],
        "676573777777600":["691","BOV_1"],
        "676673677777600":["692","BOV_1"],
        "676743677777600":["693","BOV"],
        "676743777777600":["694","BOV"],
        "676763677777600":["695","BOV"],
        "676773674177600":["696","BOV_1"],
        "676773676177600":["697","BOV_1"],
        "676773677777200":["698","BOV_1"],
        "677777777720571":["699","H3"],
        "700741007764671":["700","AFRI"],
        "700741007774671":["701","AFRI"],
        "700775747413771":["702","EAI6-BGD1"],
        "703177400001771":["703","CAS1-Kili"],
        "703777740003111":["704","CAS1-Delhi"],
        "717377777760771":["705","T1"],
        "730177577760771":["706","T1"],
        "736377777760771":["707","S"],
        "737776777760771":["708","X1"],
        "737777607760700":["709","LAM4"],
        "737777700000031":["710","None"],
        "737777777413731":["711","EAI1-SOM"],
        "747777777760731":["712","T2"],
        "763777777760771":["713","T1"],
        "777777006720771":["714","H3"],
        "770477777777671":["715","AFRI_1"],
        "772777400001671":["716","None"],
        "774177777760771":["717","T1"],
        "776000000000771":["718","None"],
        "776177407760771":["719","LAM3"],
        "776177607760371":["720","LAM3"],
        "776377777760371":["721","S"],
        "777700007413731":["722","EAI1-SOM"],
        "777700074413771":["723","EAI4-VNM"],
        "777701002001731":["724","None"],
        "777737607760770":["725","LAM5"],
        "777737747413771":["726","EAI6-BGD1"],
        "777737774020731":["727","H1"],
        "777737777760431":["728","T3"],
        "657777607760771":["729","LAM1"],
        "777757777413731":["730","EAI1-SOM"],
        "777761607760771":["731","LAM9"],
        "777763777760771":["732","T1"],
        "777775747413771":["733","EAI6-BGD1"],
        "777776777411731":["734","EAI1-SOM"],
        "777776777413731":["735","EAI1-SOM"],
        "777777577760731":["736","T2"],
        "777777607760571":["737","LAM9"],
        "777777607760760":["738","LAM9"],
        "777777707413771":["739","EAI5"],
        "777777747720771":["740","H3"],
        "777777757720771":["741","H3"],
        "777777770020771":["742","H"],
        "777777776000071":["743","None"],
        "777777777411731":["744","EAI1-SOM"],
        "777777777413131":["745","EAI1-SOM"],
        "777777777520771":["746","H3"],
        "777777777720471":["747","H3"],
        "777777777720760":["748","H3"],
        "777777677777771":["749","None"],
        "037677500020771":["750","H3"],
        "077777777760771":["751","T"],
        "440013777777600":["752","BOV"],
        "477777607760771":["753","LAM1"],
        "503777740003771":["754","CAS1-Delhi"],
        "577777677720771":["755","H3"],
        "673777477413771":["756","EAI2-Manila"],
        "676573777677600":["757","BOV_1"],
        "677777477413700":["758","EAI2-Manila"],
        "077637777760771":["759","T1"],
        "774037000020700":["760","None"],
        "774077007777071":["761","AFRI_3"],
        "774777777420731":["762","H3"],
        "777700777413700":["763","EAI5"],
        "777757777720771":["764","H3"],
        "777760077760771":["765","T1"],
        "777761007760771":["766","LAM9"],
        "777773777413700":["767","EAI"],
        "777777407720771":["768","H3"],
        "777777474020771":["769","H1"],
        "777677607760771":["770","LAM9"],
        "777777717760771":["771","T1"],
        "777777743460771":["772","LAM10-CAM"],
        "777777760000031":["773","None"],
        "777777773760771":["774","T1"],
        "777777777000371":["775","None"],
        "777777777410771":["776","EAI5"],
        "777777777420771":["777","H3"],
        "676573777077600":["778","BOV_1"],
        "777777717720771":["779","H3"],
        "777777777600771":["780","None"],
        "000000004020731":["781","H2"],
        "417777607760771":["782","LAM1"],
        "477777777760731":["783","T2"],
        "776377777760731":["784","S"],
        "777777554760771":["785","T5-Madrid2"],
        "777777760000000":["786","None"],
        "777777777760071":["787","T1"],
        "577777002060771":["788","None"],
        "676377777760771":["789","S"],
        "776377777740771":["790","S"],
        "777777760020771":["791","H3"],
        "763777777413771":["792","EAI5"],
        "577736777760601":["793","X2"],
        "703757740003771":["794","CAS1-Delhi"],
        "000000006760771":["795","LAM3"],
        "000000000001771":["796","Beijing"],
        "676773777777200":["797","BOV_1"],
        "437777777760771":["798","T1"],
        "770767677777671":["799","AFRI_1"],
        "774000370020771":["800","H3"],
        "774777777760771":["801","T1"],
        "777603000000011":["802","ZERO"],
        "777740007760771":["803","LAM9"],
        "477777777760771":["804","T1"],
        "737417606060731":["805","LAM11-ZWE"],
        "757777777413731":["806","EAI1-SOM"],
        "770077606060731":["807","LAM11-ZWE"],
        "777747606060771":["808","LAM11-ZWE"],
        "777775606060771":["809","LAM11-ZWE"],
        "777777602060771":["810","T1"],
        "777777604060731":["811","LAM11-ZWE"],
        "777777606060571":["812","LAM11-ZWE"],
        "777777606060631":["813","LAM11-ZWE"],
        "777777606060671":["814","LAM11-ZWE"],
        "777777606060731":["815","LAM11-ZWE"],
        "077777606060771":["816","LAM11-ZWE"],
        "777777777420731":["817","H3"],
        "200003770000000":["818","BOV_4-Caprae"],
        "676763677777400":["819","BOV"],
        "676763777777600":["820","BOV"],
        "674000000000171":["821","None"],
        "757777607760771":["822","LAM9"],
        "776000003760771":["823","T1"],
        "777777457760771":["824","T-H37Rv_var2"],
        "777777600000171":["825","None"],
        "677737207760771":["826","LAM2"],
        "776337770760771":["827","S"],
        "377777607760731":["828","LAM4"],
        "640003777777600":["829","BOV"],
        "000000007720731":["830","H3"],
        "776367777760771":["831","S"],
        "777727777760731":["832","T"],
        "757777777760700":["833","T1"],
        "777767777740771":["834","T1"],
        "777767777720671":["835","H3"],
        "777777777700131":["836","None"],
        "777777777020731":["837","H"],
        "777777743760751":["838","LAM10-CAM"],
        "777777743700171":["839","None"],
        "777775777720771":["840","H3"],
        "777767643760771":["841","LAM10-CAM"],
        "777766770000000":["842","None"],
        "777177777720771":["843","H3"],
        "761777743760731":["844","LAM10-CAM"],
        "761767743760731":["845","LAM10-CAM"],
        "760777777760731":["846","T2"],
        "760001743760771":["847","LAM10-CAM"],
        "737777777760731":["848","T2"],
        "637777777720771":["849","H3"],
        "437777743760771":["850","LAM10-CAM"],
        "403777702000371":["851","None"],
        "400003743760771":["852","LAM10-CAM"],
        "377777777760731":["853","T2"],
        "177777403760771":["854","T1"],
        "037767777720731":["855","H3"],
        "774077770005071":["856","AFRI_3"],
        "774067770007071":["857","AFRI_3"],
        "774067770005071":["858","AFRI_3"],
        "770777777747071":["859","AFRI"],
        "574067607777071":["860","AFRI_2"],
        "574067606767071":["861","AFRI_2"],
        "777767777360731":["862","T2"],
        "000000007400000":["863","PINI2"],
        "703740000000760":["864","CAS"],
        "477777677413071":["865","EAI3-IND"],
        "577777607760771":["866","LAM9"],
        "777737607560731":["867","LAM"],
        "777676777760771":["868","X1"],
        "777774047560771":["869","T1"],
        "777767776760731":["870","T2"],
        "737777777720771":["871","H3"],
        "773777623760771":["872","T1"],
        "703400000000000":["873","CAS"],
        "776377777760431":["874","S"],
        "777717777760731":["875","T2"],
        "703777740000731":["876","CAS"],
        "770000770000000":["877","None"],
        "777776777760571":["878","X1"],
        "777767777760671":["879","T1"],
        "777777000760771":["880","T1"],
        "776377770000731":["881","None"],
        "577777757413771":["882","EAI6-BGD1"],
        "777777754020771":["883","H1"],
        "376377777760771":["884","S"],
        "777737607460771":["885","LAM5"],
        "777776776760601":["886","X2"],
        "557767777720771":["887","H3"],
        "777777777760631":["888","T2"],
        "757777757413771":["889","EAI6-BGD1"],
        "741774047560771":["890","T1"],
        "777777607660771":["891","LAM9"],
        "777777477413771":["892","EAI5"],
        "777777017760771":["893","T"],
        "677777477413731":["894","EAI2-Manila"],
        "677777477013771":["895","EAI2-Manila"],
        "777734077560771":["896","T3"],
        "676003477413771":["897","EAI2-Manila"],
        "677777774020771":["898","H1"],
        "670000777760771":["899","T1-RUS2"],
        "677777607740011":["900","Ambiguous:LAM1 LAM4"],
        "700036777740031":["901","X3"],
        "776137607760631":["902","LAM3"],
        "777772777760601":["903","X2"],
        "775775757760771":["904","T5"],
        "677737607700771":["905","None"],
        "777774177720771":["906","H3"],
        "777777677760770":["907","T1"],
        "677737607760700":["908","LAM2"],
        "740777676000011":["909","None"],
        "700000007177771":["910","None"],
        "777777600060771":["911","None"],
        "677777777413371":["912","EAI5"],
        "777743777760771":["913","T1"],
        "776377777720771":["914","None"],
        "761777777720731":["915","H3"],
        "777777607760411":["916","LAM4"],
        "777577777760771":["917","T1"],
        "770000607760771":["918","None"],
        "777777774020760":["919","H1"],
        "777633777420771":["920","H3"],
        "437737777420771":["921","H4"],
        "774747377420771":["922","H3"],
        "676377777720771":["923","H3"],
        "777600007413371":["924","EAI5"],
        "777717777720771":["925","H3"],
        "773777777760771":["926","T1"],
        "440060000000771":["927","None"],
        "777777750060771":["928","T"],
        "777737777620771":["929","H3"],
        "677777404760771":["930","LAM7-TUR"],
        "774000007760771":["931","LAM3"],
        "777777663760731":["932","T2"],
        "777777607760071":["933","LAM9"],
        "777777777413071":["934","EAI5"],
        "775777777720771":["935","H3"],
        "740737677760731":["936","Ambiguous:T3 T2"],
        "777777777013771":["937","EAI5"],
        "474177777413771":["938","EAI5"],
        "775777777413771":["939","EAI5"],
        "000000000003401":["940","Beijing"],
        "000000000003751":["941","Beijing"],
        "777777770760731":["942","T2"],
        "777777757760731":["943","Ambiguous:T2 T5"],
        "777777774000071":["944","Ambiguous:T2 T5"],
        "763743777413771":["945","EAI5"],
        "777777740020771":["946","H"],
        "777777777413631":["947","EAI5"],
        "777777760020611":["948","H3"],
        "777777017720771":["949","H3"],
        "700037607760771":["950","LAM3"],
        "774177777413771":["951","EAI5"],
        "603777740003771":["952","CAS1-Delhi"],
        "777377577360771":["953","T1"],
        "703677740003771":["954","CAS1-Delhi"],
        "000000007777771":["955","None"],
        "777777777760011":["956","T1"],
        "676501777177600":["957","BOV"],
        "777777776013731":["958","EAI1-SOM"],
        "777741777413731":["959","EAI1-SOM"],
        "477737607760771":["960","LAM2"],
        "407777607760771":["961","LAM1"],
        "777777777413031":["962","EAI5"],
        "777777737720771":["963","H3"],
        "437777607760771":["964","LAM1"],
        "437777407760771":["965","LAM1"],
        "775777777760771":["966","T1"],
        "000000006177600":["967","PINI2"],
        "000001705777600":["968","BOV"],
        "000001707777000":["969","BOV"],
        "000001707777600":["970","BOV"],
        "000001747777000":["971","BOV"],
        "000001747777600":["972","BOV"],
        "016773777777600":["973","BOV"],
        "046773777764600":["974","BOV"],
        "046773777767600":["975","BOV"],
        "076773777777400":["976","BOV"],
        "076773777777600":["977","BOV"],
        "200003777357600":["978","BOV_4-Caprae"],
        "200003777777600":["979","BOV"],
        "400000077777600":["980","BOV"],
        "416773777777000":["981","BOV"],
        "416773777777600":["982","BOV"],
        "600373777777600":["983","BOV"],
        "602753377777600":["984","BOV_1"],
        "606773677747600":["985","BOV_1"],
        "606773777777600":["986","BOV_1"],
        "614173777777600":["987","BOV"],
        "616601777777600":["988","BOV"],
        "616733777777600":["989","BOV_1"],
        "616772777777600":["990","BOV_1"],
        "616773737777600":["991","BOV_1"],
        "616773777677600":["992","BOV_1"],
        "616773777757600":["993","BOV_1"],
        "616773777763600":["994","BOV_1"],
        "616773777775600":["995","BOV_1"],
        "636373677777600":["996","BOV"],
        "636773777777600":["997","BOV_1"],
        "644773777767600":["998","BOV"],
        "646770077767600":["999","BOV"],
        "646773776167600":["1000","BOV_1"],
        "646773777767200":["1001","BOV_1"],
        "646773777767600":["1002","BOV_1"],
        "656773777777600":["1003","BOV_1"],
        "662773777777600":["1004","BOV_1"],
        "672773777775600":["1005","BOV_1"],
        "674373777774000":["1006","BOV"],
        "674373777777600":["1007","BOV"],
        "676563777777600":["1008","BOV"],
        "676713677767600":["1009","BOV_1"],
        "676733677777600":["1010","BOV_1"],
        "676743377777600":["1011","BOV"],
        "676760000007600":["1012","BOV"],
        "676760006067600":["1013","BOV"],
        "676763637767600":["1014","BOV"],
        "676770077777600":["1015","BOV"],
        "676772777767600":["1016","BOV_1"],
        "676773017737600":["1017","BOV_1"],
        "676773377757600":["1018","BOV_1"],
        "676773377777600":["1019","BOV_1"],
        "676773417777600":["1020","BOV_1"],
        "676773674037600":["1021","BOV_1"],
        "676773677743600":["1022","BOV_1"],
        "676773677757600":["1023","BOV_1"],
        "676773677761600":["1024","BOV_1"],
        "676773677763600":["1025","BOV_1"],
        "676773677774600":["1026","BOV_1"],
        "676773677775600":["1027","BOV_1"],
        "676773677777000":["1028","BOV_1"],
        "676773737766600":["1029","BOV_1"],
        "676773737777600":["1030","BOV_1"],
        "676773764146600":["1031","BOV_1"],
        "676773776003600":["1032","BOV_1"],
        "676773777075600":["1033","BOV_1"],
        "676773777600600":["1034","BOV_1"],
        "676773777627600":["1035","BOV_1"],
        "676773777637600":["1036","BOV_1"],
        "676773777677600":["1037","BOV_1"],
        "676773777735600":["1038","BOV_1"],
        "676773777736600":["1039","BOV_1"],
        "676773777737600":["1040","BOV_1"],
        "676773777743600":["1041","BOV_1"],
        "676773777753600":["1042","BOV_1"],
        "676773777766600":["1043","BOV_1"],
        "676773777767600":["1044","BOV_1"],
        "676773777770600":["1045","BOV_1"],
        "676773777771600":["1046","BOV_1"],
        "676773777773600":["1047","BOV_1"],
        "676773777775600":["1048","BOV_1"],
        "777637547013771":["1049","EAI5"],
        "777730000000771":["1050","None"],
        "777777437760771":["1051","T1"],
        "776367377760771":["1052","S"],
        "777767637760771":["1053","T1"],
        "777737770020731":["1054","H3"],
        "777767770020720":["1055","H3"],
        "777477777760731":["1056","T2"],
        "000003743760771":["1057","LAM10-CAM"],
        "037777777760731":["1058","T2"],
        "777777774417771":["1059","None"],
        "777777776160771":["1060","T1"],
        "757776777760771":["1061","X1"],
        "700774007413771":["1062","EAI5"],
        "776377777660771":["1063","S"],
        "771777607760771":["1064","LAM9"],
        "777717743760771":["1065","LAM10-CAM"],
        "777767607560771":["1066","LAM6"],
        "774037707760700":["1067","None"],
        "776337377760771":["1068","S"],
        "771777777760771":["1069","T1"],
        "777777607760371":["1070","LAM9"],
        "777771607760771":["1071","LAM9"],
        "777731607760771":["1072","LAM5"],
        "777637777760771":["1073","T1"],
        "777777607760761":["1074","LAM9"],
        "777770607760771":["1075","LAM9"],
        "007777607760771":["1076","LAM1"],
        "777777377760731":["1077","Ambiguous:T4 T2"],
        "377703377760731":["1078","Ambiguous:T4 T2"],
        "577777703760771":["1079","T1"],
        "777776777760701":["1080","X1"],
        "777736777760731":["1081","Ambiguous:T3 T2 X1"],
        "777777777413311":["1082","EAI5"],
        "777700000001731":["1083","None"],
        "600000000360771":["1084","T1"],
        "777777774620731":["1085","H3"],
        "740737607760731":["1086","Ambiguous:LAM5 LAM4"],
        "177777603760771":["1087","T1"],
        "777767777763771":["1088","Manu2"],
        "403777740003771":["1089","None"],
        "077777777413771":["1090","EAI5"],
        "703777740002771":["1091","CAS1-Delhi"],
        "702777740003771":["1092","CAS1-Delhi"],
        "703777600003771":["1093","CAS"],
        "777777777563771":["1094","Manu2"],
        "000076777760771":["1095","X1"],
        "577777777763771":["1096","Manu2"],
        "400177777413071":["1097","EAI3-IND"],
        "777777777760331":["1098","T2"],
        "776000017760471":["1099","T1"],
        "037776777760601":["1100","X2"],
        "000076700000031":["1101","None"],
        "617776777760401":["1102","X2"],
        "770000757760771":["1103","T1-RUS2"],
        "777777757660771":["1104","T5"],
        "777773777760771":["1105","T1"],
        "777777607760400":["1106","LAM4"],
        "777700007760771":["1107","LAM"],
        "200003370000000":["1108","BOV_4-Caprae"],
        "200003400037600":["1109","BOV_4-Caprae"],
        "376037777760771":["1110","T1"],
        "377777777760760":["1111","T1"],
        "403777777760771":["1112","T1"],
        "476377777760771":["1113","S"],
        "477377777760731":["1114","T2"],
        "577777777700000":["1115","None"],
        "607777777720771":["1116","H3"],
        "674777777420771":["1117","H3"],
        "676673757777600":["1118","BOV_1"],
        "700036777760471":["1119","X3"],
        "703760000000331":["1120","CAS"],
        "717777777760611":["1121","T1"],
        "767777777760771":["1122","T1"],
        "770000777740171":["1123","T1-RUS2"],
        "770003777720771":["1124","H3"],
        "773737774020771":["1125","H1"],
        "775677757760771":["1126","T5"],
        "776377577760771":["1127","S"],
        "776377730360771":["1128","S"],
        "776777777760771":["1129","T1"],
        "777377777700000":["1130","None"],
        "777617347760471":["1131","T4-CEU1"],
        "777637777720771":["1132","H3"],
        "777737347760471":["1133","T4-CEU1"],
        "777737777420731":["1134","H3"],
        "777737777720731":["1135","H3"],
        "777740004720771":["1136","H3"],
        "777740015760771":["1137","T1"],
        "777767774760771":["1138","T1"],
        "777777374020771":["1139","H1"],
        "777777404120771":["1140","H3"],
        "777777741760771":["1141","T1"],
        "777777777720071":["1142","H3"],
        "777777777720371":["1143","H3"],
        "777777600760771":["1144","T1"],
        "677777607060771":["1145","LAM1"],
        "741737607560771":["1146","Ambiguous:T3-OSA LAM6"],
        "777777600560771":["1147","T1"],
        "777734377760771":["1148","T3"],
        "777777777777331":["1149","None"],
        "700036776360771":["1150","X3"],
        "700377740001771":["1151","CAS"],
        "000037777760770":["1152","T1"],
        "772700000003631":["1153","None"],
        "777777607760751":["1154","LAM9"],
        "577777774020771":["1155","H1"],
        "636377607760771":["1156","LAM1"],
        "700076777760770":["1157","X3"],
        "640013377777600":["1158","BOV_3"],
        "437777777720771":["1159","H3"],
        "777737207760771":["1160","LAM5"],
        "777740007760531":["1161","LAM4"],
        "000000000002171":["1162","Beijing"],
        "677737777760771":["1163","T3"],
        "637777777700371":["1164","None"],
        "777737757420771":["1165","H3"],
        "777377777760771":["1166","T1"],
        "777703737760771":["1167","T1"],
        "000000000003631":["1168","Beijing"],
        "677737477413771":["1169","EAI2-Manila"],
        "676177477413771":["1170","EAI2-Manila"],
        "677777477412771":["1171","EAI2-Manila"],
        "000000000403771":["1172","None"],
        "770000777760731":["1173","T1-RUS2"],
        "774777757420771":["1174","H3"],
        "774777777420371":["1175","H3"],
        "777757607760771":["1176","LAM9"],
        "777777767700000":["1177","None"],
        "777771000000731":["1178","None"],
        "777776776360771":["1179","X1"],
        "763777777413700":["1180","EAI5"],
        "676773757777600":["1181","BOV_1"],
        "477777777413731":["1182","EAI1-SOM"],
        "777777777413331":["1183","EAI1-SOM"],
        "000002000003771":["1184","None"],
        "676673777777600":["1185","BOV_1"],
        "777777777403671":["1186","None"],
        "674000003403771":["1187","None"],
        "777777777003771":["1188","None"],
        "677777477403771":["1189","EAI2-Manila"],
        "777700776413771":["1190","EAI5"],
        "777737776760731":["1191","Ambiguous:T3 T2"],
        "777777677763771":["1192","Manu2"],
        "777777677773771":["1193","Manu1"],
        "777777677770771":["1194","Manu3"],
        "777767477763771":["1195","Manu2"],
        "777777777761771":["1196","None"],
        "703777740203171":["1197","None"],
        "703737740003171":["1198","CAS1-Delhi"],
        "703777740001171":["1199","CAS1-Delhi"],
        "703777747777771":["1200","None"],
        "677636207760771":["1201","LAM2"],
        "000000003760771":["1202","None"],
        "703777640003171":["1203","CAS1-Delhi"],
        "777770000000731":["1204","None"],
        "777777774020671":["1205","H1"],
        "437777777400171":["1206","None"],
        "737777777700771":["1207","None"],
        "000000104020771":["1208","H1"],
        "000000377760771":["1209","T1"],
        "017777607760771":["1210","LAM1"],
        "576377777760771":["1211","S"],
        "600000000060751":["1212","T1"],
        "647737607760771":["1213","LAM2"],
        "777617777760771":["1214","T3"],
        "677735607760771":["1215","LAM"],
        "677737607700371":["1216","None"],
        "677777477413151":["1217","EAI2-Manila"],
        "700076777740771":["1218","X3"],
        "737777423560771":["1219","T1"],
        "754377607760700":["1220","LAM"],
        "757767777760771":["1221","T1"],
        "770177607760771":["1222","LAM3"],
        "776077777760771":["1223","T1"],
        "776177607760700":["1224","LAM3"],
        "776377700160771":["1225","S"],
        "777617777777771":["1226","None"],
        "777737557760771":["1227","T5-Madrid2"],
        "777740017760731":["1228","T2"],
        "777743777720771":["1229","H3"],
        "777757774020771":["1230","H1"],
        "777774057760771":["1231","T5"],
        "777774077560731":["1232","T2"],
        "777777077760771":["1233","T1"],
        "777777644720771":["1234","H3"],
        "777777744720771":["1235","H3"],
        "777777760000611":["1236","None"],
        "777777763660771":["1237","T1"],
        "777777776720771":["1238","H3"],
        "777777777160031":["1239","T1"],
        "777760001760771":["1240","T5-RUS1"],
        "777777607700771":["1241","LAM9"],
        "776377777420771":["1242","H3"],
        "777377777720771":["1243","H3"],
        "776377777720731":["1244","H3"],
        "776377777400071":["1245","None"],
        "777776777720771":["1246","H3"],
        "777777607763771":["1247","Manu2"],
        "760003777760771":["1248","T1"],
        "003777607760771":["1249","LAM1"],
        "777777737740771":["1250","T1"],
        "777767777413731":["1251","EAI1-SOM"],
        "777777177760771":["1252","T1"],
        "676377737760771":["1253","S"],
        "777776777060771":["1254","X1"],
        "177777657760771":["1255","T5"],
        "770177774020771":["1256","H1"],
        "777736000000000":["1257","None"],
        "707777347760471":["1258","T4-CEU1"],
        "777740000160771":["1259","T1"],
        "000000007620771":["1260","H3"],
        "777777404760731":["1261","LAM7-TUR"],
        "000000007000771":["1262","None"],
        "477777377410771":["1263","EAI8-MDG"],
        "703777740000000":["1264","CAS1-Delhi"],
        "776777777760731":["1265","T2"],
        "700777740003771":["1266","CAS"],
        "774037777760300":["1267","T1"],
        "777760057760771":["1268","T5"],
        "574777777420771":["1269","H4"],
        "757776777760601":["1270","X2"],
        "776377677760771":["1271","S"],
        "577737777760571":["1272","T3"],
        "700076777760731":["1273","X3"],
        "777777760000071":["1274","None"],
        "777777774000371":["1275","None"],
        "777017777420771":["1276","H3"],
        "777777207760771":["1277","LAM9"],
        "037677777760771":["1278","T1"],
        "737777757760771":["1279","T5"],
        "000000007760770":["1280","LAM3"],
        "777537777420771":["1281","H3"],
        "777777400020771":["1282","H3"],
        "777777777020771":["1283","H3"],
        "741703777760771":["1284","T1"],
        "577776755412731":["1285","Ambiguous:EAI6-BGD1 EAI1-SOM"],
        "474177777613771":["1286","Manu1"],
        "400037777613771":["1287","Manu1"],
        "777477607763771":["1288","Manu2"],
        "777777717763771":["1289","Manu2"],
        "777777607723771":["1290","Manu2"],
        "777737777423771":["1291","Manu2"],
        "674777377420771":["1292","H3"],
        "676177607760771":["1293","LAM3"],
        "774177407760771":["1294","LAM3"],
        "776163607760771":["1295","LAM3"],
        "776371777760771":["1296","S"],
        "664073077777600":["1297","BOV_2"],
        "642217777777600":["1298","BOV"],
        "440003777777600":["1299","BOV"],
        "001776777760601":["1300","X2"],
        "577777777760601":["1301","T1"],
        "577777777760731":["1302","T2"],
        "624073777777600":["1303","BOV"],
        "640013737777600":["1304","BOV_3"],
        "064063777777600":["1305","BOV"],
        "264063777777600":["1306","BOV"],
        "064061777777600":["1307","BOV"],
        "664063777700600":["1308","BOV"],
        "064263777777600":["1309","BOV"],
        "464223777774600":["1310","BOV"],
        "000000000003700":["1311","Beijing"],
        "703777740003131":["1312","CAS1-Delhi"],
        "777777775413731":["1313","EAI1-SOM"],
        "703377740003171":["1314","CAS1-Delhi"],
        "700066777760671":["1315","X3"],
        "717777777413731":["1316","EAI1-SOM"],
        "577767777760671":["1317","T1"],
        "577767777760771":["1318","T1"],
        "500367740003771":["1319","CAS2"],
        "700177740003771":["1320","CAS2"],
        "677777607760731":["1321","Ambiguous:LAM1 LAM4"],
        "777703077760771":["1322","T1"],
        "503767740003671":["1323","CAS1-Delhi"],
        "777760047760771":["1324","T1"],
        "777761050000771":["1325","None"],
        "777037777413700":["1326","EAI5"],
        "703637740003771":["1327","CAS1-Delhi"],
        "777777034020771":["1328","H1"],
        "777776777560771":["1329","X1"],
        "377777757760771":["1330","T5"],
        "776175407760771":["1331","LAM3"],
        "677777777760731":["1332","T2"],
        "776370001760771":["1333","S"],
        "776370001760731":["1334","S"],
        "776177607760400":["1335","LAM3"],
        "676377717760471":["1336","S"],
        "777700607760771":["1337","LAM5"],
        "777777777777671":["1338","None"],
        "677737777760731":["1339","Ambiguous:T3 T2"],
        "777775757413371":["1340","EAI6-BGD1"],
        "777776757760601":["1341","X2"],
        "477777177413071":["1342","EAI3-IND"],
        "703737740003771":["1343","CAS1-Delhi"],
        "703777440003771":["1344","CAS1-Delhi"],
        "703777700000371":["1345","None"],
        "703777640003770":["1346","CAS1-Delhi"],
        "777700777760771":["1347","T1"],
        "774037777760701":["1348","T1"],
        "057777777760771":["1349","T1"],
        "777017777700000":["1350","None"],
        "776377677700771":["1351","None"],
        "777677770000000":["1352","None"],
        "777777774000000":["1353","None"],
        "576177607760771":["1354","LAM3"],
        "777777407560731":["1355","LAM"],
        "776377777760751":["1356","S"],
        "703777400003031":["1357","CAS"],
        "777417760060771":["1358","T1"],
        "477774374020771":["1359","H1"],
        "577777777720631":["1360","H3"],
        "757737777760731":["1361","Ambiguous:T3 T2"],
        "677777700000371":["1362","None"],
        "777777777400171":["1363","None"],
        "000000000003471":["1364","Beijing"],
        "000000007413771":["1365","EAI5"],
        "000076777760671":["1366","X1"],
        "377737607760771":["1367","LAM5"],
        "701777747770371":["1368","None"],
        "477777777413671":["1369","EAI5"],
        "757777757760771":["1370","T5"],
        "767777606760771":["1371","LAM8"],
        "777577777413771":["1372","EAI5"],
        "777677777413731":["1373","EAI1-SOM"],
        "777777700003771":["1374","None"],
        "777777753413771":["1375","EAI6-BGD1"],
        "777777775413371":["1376","EAI5"],
        "777777777412671":["1377","EAI5"],
        "777777777770771":["1378","Manu3"],
        "773777777770771":["1379","Manu3"],
        "774777777410771":["1380","EAI5"],
        "774777775410771":["1381","EAI5"],
        "777777776720671":["1382","H3"],
        "773777777760731":["1383","T2"],
        "000000277420771":["1384","H4"],
        "537767776760631":["1385","T1"],
        "777777777403701":["1386","None"],
        "407760007760771":["1387","T5-RUS1"],
        "777777743413331":["1388","Ambiguous:EAI6-BGD1 EAI1-SOM"],
        "177777777413731":["1389","EAI1-SOM"],
        "777777747413371":["1390","EAI6-BGD1"],
        "777777700003371":["1391","EAI7-BGD2"],
        "777777777413431":["1392","EAI5"],
        "777777777405700":["1393","None"],
        "777776777760671":["1394","X1"],
        "474377767413771":["1395","EAI5"],
        "777777750002371":["1396","None"],
        "777777761413700":["1397","EAI5"],
        "577777740003771":["1398","None"],
        "077777777413700":["1399","EAI5"],
        "700000007413371":["1400","EAI5"],
        "703777740003671":["1401","CAS1-Delhi"],
        "577770050000000":["1402","None"],
        "777777757003371":["1403","None"],
        "777777767413731":["1404","EAI1-SOM"],
        "703743740003771":["1405","CAS1-Delhi"],
        "777777757413171":["1406","EAI6-BGD1"],
        "775377777413771":["1407","EAI5"],
        "000000007413371":["1408","EAI5"],
        "777777757413271":["1409","EAI6-BGD1"],
        "777777700000000":["1410","None"],
        "777477757413371":["1411","EAI6-BGD1"],
        "777377757413271":["1412","EAI6-BGD1"],
        "177743777413731":["1413","EAI1-SOM"],
        "777757757413371":["1414","EAI6-BGD1"],
        "077743777413731":["1415","EAI1-SOM"],
        "777747757413771":["1416","EAI6-BGD1"],
        "777777757413071":["1417","EAI6-BGD1"],
        "577777750003771":["1418","None"],
        "717777700003371":["1419","None"],
        "777777757410071":["1420","EAI6-BGD1"],
        "777777757403371":["1421","None"],
        "703777000000771":["1422","CAS1-Delhi"],
        "477774777413771":["1423","EAI5"],
        "777777010003371":["1424","EAI7-BGD2"],
        "777777757413331":["1425","Ambiguous:EAI6-BGD1 EAI1-SOM"],
        "777737777760711":["1426","T3"],
        "457001777413771":["1427","EAI5"],
        "774707777413731":["1428","EAI1-SOM"],
        "703777750003771":["1429","None"],
        "757777700003371":["1430","None"],
        "174000007413731":["1431","EAI1-SOM"],
        "777743300000000":["1432","None"],
        "577737757413771":["1433","EAI6-BGD1"],
        "773777740000171":["1434","None"],
        "774777777413731":["1435","EAI1-SOM"],
        "577777757403771":["1436","None"],
        "703737740003731":["1437","CAS1-Delhi"],
        "777776757413371":["1438","EAI6-BGD1"],
        "674373677777600":["1439","BOV"],
        "676777677777600":["1440","BOV"],
        "076777777777600":["1441","BOV"],
        "614077777777600":["1442","BOV"],
        "616777777777600":["1443","BOV"],
        "777400000060771":["1444","T1"],
        "407757607760771":["1445","LAM1"],
        "777757677420771":["1446","H3"],
        "777637777420771":["1447","H3"],
        "777637777420731":["1448","H3"],
        "777777677421771":["1449","None"],
        "777777670000771":["1450","None"],
        "777760000000000":["1451","None"],
        "774777677420771":["1452","H3"],
        "776177637763771":["1453","Manu2"],
        "774000677760371":["1454","T1"],
        "774377677420771":["1455","H3"],
        "777637677420771":["1456","H3"],
        "777757777420771":["1457","H3"],
        "773737777400771":["1458","None"],
        "777776777760700":["1459","X1"],
        "777777377763771":["1460","Manu2"],
        "773737677420771":["1461","H3"],
        "777777770003771":["1462","None"],
        "670000777760731":["1463","T1-RUS2"],
        "777377770000771":["1464","None"],
        "774077770001071":["1465","AFRI_3"],
        "757777606060771":["1466","LAM11-ZWE"],
        "477777777410771":["1467","EAI5"],
        "077777606060671":["1468","LAM11-ZWE"],
        "017776607760771":["1469","LAM1"],
        "777776607760771":["1470","LAM9"],
        "777737606060771":["1471","LAM11-ZWE"],
        "477677777410771":["1472","EAI5"],
        "700777746013771":["1473","EAI5"],
        "677777627760771":["1474","T1"],
        "777357777760771":["1475","T1"],
        "736177607700171":["1476","AFRI_2"],
        "677777463413701":["1477","EAI2-Manila"],
        "675777477413771":["1478","EAI2-Manila"],
        "674777470001771":["1479","EAI2-Manila"],
        "774776000420771":["1480","H3"],
        "774777777423771":["1481","Manu2"],
        "775740003763771":["1482","Manu2"],
        "777717377760771":["1483","T4"],
        "777777774023771":["1484","Manu2"],
        "777740007763771":["1485","Manu2"],
        "300036777760771":["1486","X3"],
        "000000007777731":["1487","None"],
        "401777777410071":["1488","EAI5"],
        "637777477413750":["1489","EAI5"],
        "677767477413771":["1490","EAI2-Manila"],
        "740000007760731":["1491","None"],
        "777700000001131":["1492","None"],
        "777777603413700":["1493","EAI5"],
        "777777676760771":["1494","T1"],
        "777777737413771":["1495","EAI5"],
        "777777756413771":["1496","EAI6-BGD1"],
        "777777775413671":["1497","EAI5"],
        "777777776000371":["1498","None"],
        "677003477413771":["1499","EAI2-Manila"],
        "677777477400771":["1500","None"],
        "677777477413671":["1501","EAI2-Manila"],
        "477757777013071":["1502","EAI3-IND"],
        "777747757413271":["1503","EAI6-BGD1"],
        "777747777413671":["1504","EAI5"],
        "777757607760671":["1505","LAM9"],
        "777757757413271":["1506","EAI6-BGD1"],
        "777757757413771":["1507","EAI6-BGD1"],
        "777757777413771":["1508","EAI5"],
        "677777472000000":["1509","None"],
        "677777476003771":["1510","None"],
        "677577477413771":["1511","EAI2-Manila"],
        "657777477413701":["1512","EAI2-Manila"],
        "777577757413771":["1513","EAI6-BGD1"],
        "757777637760771":["1514","T1"],
        "477777777403771":["1515","None"],
        "777377777761771":["1516","None"],
        "477777377417771":["1517","None"],
        "477777377403771":["1518","None"],
        "000037777403771":["1519","None"],
        "477777277403771":["1520","None"],
        "700774007403771":["1521","None"],
        "777737606063771":["1522","Manu2"],
        "777777667763771":["1523","Manu2"],
        "700777747403771":["1524","None"],
        "376137607760771":["1525","LAM3"],
        "700037177760771":["1526","T1"],
        "777776774020771":["1527","H1"],
        "774377607760771":["1528","LAM9"],
        "777777777720171":["1529","H3"],
        "777777607760711":["1530","LAM4"],
        "777776777760071":["1531","X1"],
        "777776775760771":["1532","X1"],
        "777763777720771":["1533","H3"],
        "777721607560731":["1534","LAM6"],
        "777577607760771":["1535","LAM9"],
        "776377607760771":["1536","LAM9"],
        "776137607760731":["1537","LAM"],
        "774137600020771":["1538","None"],
        "773777777720771":["1539","H3"],
        "377377607760771":["1540","LAM9"],
        "376377607760771":["1541","None"],
        "177776777760601":["1542","X2"],
        "076160000000071":["1543","None"],
        "777777777760531":["1544","T2"],
        "601775607760771":["1545","LAM1"],
        "077777777413011":["1546","EAI5"],
        "777727777760771":["1547","T3"],
        "777703400000360":["1548","None"],
        "775777606060731":["1549","LAM11-ZWE"],
        "377777777660760":["1550","T1"],
        "701777740003771":["1551","CAS1-Delhi"],
        "777777774020631":["1552","H1"],
        "775777401760771":["1553","T1"],
        "402777607760731":["1554","Ambiguous:LAM1 LAM4"],
        "700001403760371":["1555","T1"],
        "775777777660771":["1556","T1"],
        "777677774020771":["1557","H1"],
        "777777775660771":["1558","T1"],
        "777777400000171":["1559","None"],
        "703777777760771":["1560","T1"],
        "777777774020371":["1561","H1"],
        "777777377400171":["1562","None"],
        "777777636760771":["1563","T1"],
        "777776737760571":["1564","X1"],
        "777740077760771":["1565","T1"],
        "747777777760771":["1566","T1"],
        "777777707560771":["1567","T1"],
        "774777377420771":["1568","H3"],
        "700037000020700":["1569","H3"],
        "777740000000000":["1570","None"],
        "777775774413771":["1571","EAI4-VNM"],
        "677777777760730":["1572","T2-Uganda"],
        "777777177413371":["1573","EAI5"],
        "760000007760771":["1574","LAM3"],
        "777777347760400":["1575","T4"],
        "777774063560771":["1576","T1"],
        "776376003760771":["1577","S"],
        "767777777760731":["1578","T2"],
        "677777777740771":["1579","T1"],
        "777777747760771":["1580","T"],
        "577477777420771":["1581","H4"],
        "777777377760700":["1582","T4"],
        "777777777760740":["1583","T1"],
        "757777630760771":["1584","T1"],
        "777777764120771":["1585","H3"],
        "177777764020771":["1586","H1"],
        "636377770760771":["1587","S"],
        "037637607760771":["1588","LAM2"],
        "776017404760771":["1589","LAM12-Madrid1"],
        "703777340003771":["1590","CAS1-Delhi"],
        "700377740003371":["1591","CAS2"],
        "774077602000031":["1592","AFRI_2"],
        "740000000020771":["1593","H3"],
        "777400007760771":["1594","LAM5"],
        "674363677777600":["1595","BOV"],
        "200003777377200":["1596","BOV_4-Caprae"],
        "707777777760771":["1597","T1"],
        "262773777772600":["1598","BOV_1"],
        "000003777377600":["1599","BOV_4-Caprae"],
        "200003770003600":["1600","BOV_4-Caprae"],
        "676773677777400":["1601","BOV_1"],
        "674023777777600":["1602","BOV"],
        "666773777677600":["1603","BOV_1"],
        "200003377377600":["1604","BOV_4-Caprae"],
        "416773677777600":["1605","BOV"],
        "603777740000771":["1606","None"],
        "601777606060771":["1607","LAM11-ZWE"],
        "777417607760700":["1608","Ambiguous:LAM5 LAM4"],
        "777637777413771":["1609","EAI5"],
        "763777607560771":["1610","LAM6"],
        "776777777720771":["1611","H3"],
        "077577757003371":["1612","None"],
        "177777777760731":["1613","T2"],
        "777776607760700":["1614","LAM4"],
        "777417777760731":["1615","T2"],
        "003777740003771":["1616","None"],
        "700036776760771":["1617","X3"],
        "577777000000011":["1618","ZERO"],
        "000002777760731":["1619","Ambiguous:T2 X1"],
        "777777604360771":["1620","LAM9"],
        "737737777760731":["1621","Ambiguous:T3 T2"],
        "761777777760731":["1622","T2"],
        "777777677760700":["1623","T1"],
        "776177607560771":["1624","LAM"],
        "664073777700600":["1625","BOV_2"],
        "777777776760771":["1626","T1"],
        "777137777760731":["1627","Ambiguous:T3 T2"],
        "077737777413771":["1628","EAI5"],
        "777777774020751":["1629","H1"],
        "777777601760771":["1630","T1"],
        "757777777760571":["1631","T1"],
        "577777776000371":["1632","None"],
        "637403607760671":["1633","LAM2"],
        "777777777723771":["1634","Manu2"],
        "200003777363600":["1635","BOV_4-Caprae"],
        "200003774075600":["1636","BOV_4-Caprae"],
        "676773777577600":["1637","BOV_1"],
        "577767777763771":["1638","Manu2"],
        "743777607760771":["1639","LAM9"],
        "777777777720740":["1640","H3"],
        "777737177760771":["1641","T3"],
        "700017777760771":["1642","T1"],
        "777777423560771":["1643","T1"],
        "777777404020771":["1644","H1"],
        "377767777760771":["1645","T1"],
        "777737607020771":["1646","H3"],
        "777777477760731":["1647","T-H37Rv_var3"],
        "400017777413771":["1648","EAI5"],
        "757677777413731":["1649","EAI1-SOM"],
        "720001400000771":["1650","None"],
        "000000000003661":["1651","Beijing"],
        "477777774020771":["1652","H1"],
        "730000000000171":["1653","None"],
        "777674077413671":["1654","EAI5"],
        "777723777760771":["1655","T3"],
        "777767770000000":["1656","None"],
        "777777377000371":["1657","None"],
        "777777714020771":["1658","H1"],
        "777703700000000":["1659","None"],
        "777777600060731":["1660","T2"],
        "677737607760671":["1661","LAM2"],
        "600000000060731":["1662","T2"],
        "777777607520771":["1663","H3"],
        "777777600560731":["1664","T2"],
        "377400000000131":["1665","None"],
        "664063777776600":["1666","BOV"],
        "616773777777200":["1667","BOV_1"],
        "676763677776200":["1668","BOV"],
        "676763677776600":["1669","BOV"],
        "064073777777600":["1670","BOV"],
        "377767607760771":["1671","LAM9"],
        "776177775760771":["1672","T1"],
        "614373777777600":["1673","BOV"],
        "000000000003761":["1674","Beijing"],
        "703367400001771":["1675","CAS1-Kili"],
        "477777377613071":["1676","Manu1"],
        "703377400203771":["1677","None"],
        "777774377720771":["1678","H3"],
        "777767737760771":["1679","T1"],
        "475777777413071":["1680","EAI3-IND"],
        "377777774020771":["1681","H1"],
        "662773677777600":["1682","BOV_1"],
        "677777777660771":["1683","T1"],
        "776000000000000":["1684","None"],
        "776077607760771":["1685","LAM3"],
        "777740004020771":["1686","H1"],
        "777767775760771":["1687","T1"],
        "777777403760771":["1688","T1"],
        "777777607600031":["1689","None"],
        "777777777762771":["1690","Manu2"],
        "677727607760771":["1691","LAM2"],
        "417136777760731":["1692","Ambiguous:T3 T2 X1"],
        "737737607760771":["1693","LAM5"],
        "277737607760771":["1694","LAM2"],
        "677737607760770":["1695","LAM2"],
        "707737607760771":["1696","LAM5"],
        "775767607760671":["1697","LAM9"],
        "777727607700031":["1698","None"],
        "777347607760671":["1699","LAM9"],
        "753777776760771":["1700","T1"],
        "777317607760771":["1701","LAM5"],
        "037737607760771":["1702","LAM2"],
        "677737607560771":["1703","LAM2"],
        "477727607760771":["1704","LAM2"],
        "777703777460771":["1705","T1"],
        "717777607760771":["1706","LAM9"],
        "677777400760771":["1707","T1"],
        "776777607760771":["1708","LAM9"],
        "777737606760771":["1709","LAM8"],
        "777737607760731":["1710","Ambiguous:LAM5 LAM4"],
        "677337607760771":["1711","LAM2"],
        "677317607760771":["1712","LAM2"],
        "663777607760771":["1713","LAM1"],
        "777740607760771":["1714","LAM9"],
        "437777607760731":["1715","Ambiguous:LAM1 LAM4"],
        "767737675560771":["1716","T3"],
        "777776777460771":["1717","X1"],
        "677776607760771":["1718","LAM1"],
        "677337607760671":["1719","LAM2"],
        "777767774413771":["1720","EAI4-VNM"],
        "777746777760771":["1721","X1"],
        "664073677777200":["1722","BOV_2"],
        "664073770037600":["1723","BOV_2"],
        "435773037604771":["1724","None"],
        "717777757413771":["1725","EAI6-BGD1"],
        "777577777413371":["1726","EAI5"],
        "776377703760771":["1727","S"],
        "776377603760771":["1728","S"],
        "700000004177771":["1729","None"],
        "676773777747600":["1730","BOV_1"],
        "577777774413771":["1731","EAI4-VNM"],
        "577767777413731":["1732","EAI1-SOM"],
        "376774737760771":["1733","T1"],
        "077767777413731":["1734","EAI1-SOM"],
        "777777774320771":["1735","H3"],
        "777777700000371":["1736","None"],
        "777760017760771":["1737","T-Tuscany"],
        "777760063760771":["1738","T1"],
        "577777377760700":["1739","T4"],
        "467777774000071":["1740","None"],
        "600000077413771":["1741","EAI5"],
        "775777774020771":["1742","H1"],
        "767777777720631":["1743","H3"],
        "577776770000000":["1744","None"],
        "773737777760771":["1745","T3"],
        "476377737760771":["1746","S"],
        "717777757760771":["1747","T5"],
        "777757777620771":["1748","H3"],
        "777777734720771":["1749","H3"],
        "777767607760731":["1750","LAM4"],
        "700066777760771":["1751","X3"],
        "677777607740771":["1752","LAM1"],
        "700037777760771":["1753","None"],
        "777767777760601":["1754","T2"],
        "677777607560771":["1755","LAM6"],
        "700026777760771":["1756","X3"],
        "757777600000000":["1757","None"],
        "777777607540771":["1758","LAM9"],
        "772177606760771":["1759","LAM3"],
        "476177607760771":["1760","LAM3"],
        "677767777760771":["1761","T1"],
        "477777637413071":["1762","EAI3-IND"],
        "437777774320731":["1763","H3"],
        "400037777405771":["1764","None"],
        "703337400001771":["1765","CAS1-Kili"],
        "760777607760771":["1766","LAM9"],
        "777770007760771":["1767","LAM9"],
        "777777207560771":["1768","LAM6"],
        "677761607760771":["1769","LAM1"],
        "000000004020571":["1770","H2"],
        "757777744020771":["1771","H1"],
        "776377774020071":["1772","H1"],
        "777777547760471":["1773","T1"],
        "777777376760771":["1774","T4"],
        "277777477413771":["1775","EAI2-Manila"],
        "776377760000171":["1776","None"],
        "703776000000011":["1777","CAS"],
        "777770000000371":["1778","None"],
        "776477777760771":["1779","None"],
        "000000000004771":["1780","None"],
        "677777477413770":["1781","EAI2-Manila"],
        "770777607777671":["1782","AFRI_1"],
        "776377743760771":["1783","LAM10-CAM"],
        "676760777777600":["1784","BOV"],
        "770000004760771":["1785","LAM7-TUR"],
        "037647777760771":["1786","None"],
        "703777740002171":["1787","CAS1-Delhi"],
        "777703400000361":["1788","None"],
        "703777740000171":["1789","CAS"],
        "777777000360771":["1790","T1"],
        "677777607160771":["1791","None"],
        "577777607720771":["1792","H3"],
        "777777777760000":["1793","T1"],
        "277777737760731":["1794","None"],
        "777777740000371":["1795","None"],
        "000077777760771":["1796","T1"],
        "777777777660731":["1797","T2"],
        "577777744020771":["1798","H1"],
        "777777347760431":["1799","T4"],
        "777777407760771":["1800","LAM9"],
        "777777777413730":["1801","EAI1-SOM"],
        "777767777720631":["1802","H3"],
        "617777607760771":["1803","LAM1"],
        "777777577720771":["1804","H3"],
        "777777777740671":["1805","T1"],
        "677747757760771":["1806","T5"],
        "777777774020571":["1807","H1"],
        "677777757760771":["1808","T5"],
        "707777347740471":["1809","T4-CEU1"],
        "200001377376600":["1810","BOV_4-Caprae"],
        "677300004760731":["1811","LAM7-TUR"],
        "777777637720771":["1812","H3"],
        "757767774020771":["1813","H1"],
        "777777617560571":["1814","T1"],
        "677777606760771":["1815","LAM8"],
        "676773677603600":["1816","BOV_1"],
        "474563777777600":["1817","BOV"],
        "640013777763600":["1818","BOV_3"],
        "476763677777200":["1819","BOV"],
        "676773277777200":["1820","BOV_1"],
        "777347777760771":["1821","T1"],
        "777777774020770":["1822","H1"],
        "777776777760770":["1823","X1"],
        "760377777760701":["1824","T1"],
        "000000004220771":["1825","H3"],
        "777716777760771":["1826","X1"],
        "777703600020771":["1827","H3"],
        "777776207760771":["1828","LAM9"],
        "775767777760671":["1829","T1"],
        "776177207760771":["1830","LAM3"],
        "777777663740771":["1831","T1"],
        "777773607760771":["1832","LAM9"],
        "777705077560571":["1833","T1"],
        "377737777760771":["1834","T3"],
        "777703377760731":["1835","Ambiguous:T4 T2"],
        "777777007660751":["1836","LAM9"],
        "000000000203771":["1837","None"],
        "777367607760771":["1838","LAM9"],
        "777327607760770":["1839","LAM5"],
        "776327777760771":["1840","S"],
        "776177607740771":["1841","LAM3"],
        "476763777777600":["1842","BOV"],
        "740777777760731":["1843","T2"],
        "777763607760771":["1844","LAM9"],
        "000173777777600":["1845","BOV"],
        "640013776777600":["1846","BOV_3"],
        "676773677377600":["1847","BOV_1"],
        "676740677777600":["1848","BOV"],
        "674573777777600":["1849","BOV"],
        "664073776177600":["1850","BOV_2"],
        "464073777777600":["1851","BOV"],
        "676773770003600":["1852","BOV_1"],
        "674773674377200":["1853","BOV"],
        "676773770000600":["1854","BOV_1"],
        "664073777773600":["1855","BOV_2"],
        "664072777777600":["1856","BOV_2"],
        "676573777177600":["1857","BOV_1"],
        "264071777777600":["1858","BOV"],
        "624073777700600":["1859","BOV"],
        "664073777741600":["1860","BOV_2"],
        "664073777707600":["1861","BOV_2"],
        "640000037777600":["1862","BOV"],
        "664073777577600":["1863","BOV_2"],
        "400027777413671":["1864","EAI5"],
        "757767777413631":["1865","EAI5"],
        "477776377413771":["1866","EAI5"],
        "770777707777651":["1867","AFRI_1"],
        "677777477413761":["1868","EAI2-Manila"],
        "777700000000000":["1869","None"],
        "077777607760730":["1870","Ambiguous:LAM1 LAM4"],
        "700037750060771":["1871","T5"],
        "777777774060771":["1872","T1"],
        "777774606060731":["1873","LAM11-ZWE"],
        "776370001760331":["1874","S"],
        "477001777413771":["1875","EAI5"],
        "740077777777671":["1876","AFRI"],
        "737377777760771":["1877","T1"],
        "703777740003600":["1878","CAS1-Delhi"],
        "777737400413731":["1879","EAI1-SOM"],
        "037603607400011":["1880","None"],
        "077777377413771":["1881","EAI8-MDG"],
        "703776600003771":["1882","CAS"],
        "703777740003760":["1883","CAS1-Delhi"],
        "777774777000171":["1884","None"],
        "037637377760771":["1885","T4"],
        "637777477413771":["1886","EAI5"],
        "703777707770371":["1887","None"],
        "677777707760771":["1888","T1"],
        "007777707760771":["1889","T1"],
        "777777707760731":["1890","T2"],
        "777777773720771":["1891","H3"],
        "777777660000171":["1892","None"],
        "777777577620771":["1893","H3"],
        "747777607760771":["1894","LAM9"],
        "777777607460731":["1895","LAM4"],
        "777637777770711":["1896","Manu3"],
        "777777770203371":["1897","None"],
        "777777770003371":["1898","EAI7-BGD2"],
        "677640007760771":["1899","LAM1"],
        "577777777413770":["1900","EAI5"],
        "777777714413771":["1901","EAI4-VNM"],
        "707777774413731":["1902","EAI4-VNM"],
        "737777774413771":["1903","EAI4-VNM"],
        "037676000000371":["1904","None"],
        "777777777460771":["1905","T1"],
        "777777605560771":["1906","LAM6"],
        "777777777774771":["1907","None"],
        "777777776020771":["1908","H3"],
        "775777777760700":["1909","T1"],
        "677737707760771":["1910","T3"],
        "677737707760770":["1911","T3"],
        "775767757760771":["1912","T5"],
        "777767757760771":["1913","T5"],
        "377777607560771":["1914","LAM6"],
        "776377737760771":["1915","S"],
        "777701777760771":["1916","T1"],
        "577777777760770":["1917","T1"],
        "776377777700771":["1918","None"],
        "777777377720731":["1919","H3"],
        "703777770001771":["1920","None"],
        "577777407560731":["1921","LAM6"],
        "777702777760771":["1922","X1"],
        "757777777700771":["1923","None"],
        "577747404760671":["1924","LAM7-TUR"],
        "754100777677761":["1925","None"],
        "777777777760701":["1926","T1"],
        "777737764020771":["1927","H1"],
        "757777776000771":["1928","None"],
        "777777777760300":["1929","T1"],
        "477777737760771":["1930","T1"],
        "767000007720731":["1931","H3"],
        "676733777777600":["1932","BOV_1"],
        "777777607760770":["1933","LAM9"],
        "777777724020771":["1934","H1"],
        "677767607760671":["1935","LAM1"],
        "577677077760771":["1936","T"],
        "777777404760760":["1937","LAM7-TUR"],
        "776377777760770":["1938","S"],
        "777777775720731":["1939","H3"],
        "503777000003771":["1940","CAS"],
        "000000000003071":["1941","Beijing"],
        "703377740003771":["1942","CAS1-Delhi"],
        "477777777613071":["1943","Manu1"],
        "400077777413071":["1944","EAI3-IND"],
        "503767740003171":["1945","CAS1-Delhi"],
        "777777777412700":["1946","EAI5"],
        "703771740003771":["1947","CAS1-Delhi"],
        "477777777403071":["1948","EAI3-IND"],
        "703777700000771":["1949","CAS"],
        "703777747770771":["1950","Manu3"],
        "703777000003771":["1951","CAS1-Delhi"],
        "777777774000771":["1952","None"],
        "777776774020601":["1953","H1"],
        "770377777777671":["1954","AFRI"],
        "777737777761771":["1955","None"],
        "467777777413071":["1956","EAI3-IND"],
        "477777777013771":["1957","EAI5"],
        "477777777513471":["1958","Manu1"],
        "703773740003771":["1959","CAS1-Delhi"],
        "333765456234331":["1960","SIT missing"],
        "417777777413071":["1961","EAI3-IND"],
        "700000000003771":["1962","CAS"],
        "703774340003771":["1963","CAS1-Delhi"],
        "477763777413071":["1964","EAI3-IND"],
        "703777740003770":["1965","CAS1-Delhi"],
        "477777777413011":["1966","EAI5"],
        "703777740000031":["1967","CAS"],
        "703777740003711":["1968","CAS1-Delhi"],
        "777777776413700":["1969","EAI5"],
        "777776757413771":["1970","EAI6-BGD1"],
        "400000040003771":["1971","None"],
        "703777740000600":["1972","CAS"],
        "703737740002771":["1973","CAS1-Delhi"],
        "777771070000771":["1974","None"],
        "577777763420771":["1975","H4"],
        "703777747763771":["1976","Manu2"],
        "477777037413071":["1977","EAI3-IND"],
        "700373740003771":["1978","CAS2"],
        "777777777400771":["1979","None"],
        "700377740000771":["1980","CAS"],
        "103777740003171":["1981","CAS1-Delhi"],
        "777777777400031":["1982","None"],
        "474000377413031":["1983","EAI3-IND"],
        "400343436742200":["1984","BOV"],
        "577746616560601":["1985","X2"],
        "777777770360771":["1986","T1"],
        "777777777760231":["1987","T1"],
        "777777604020731":["1988","H1"],
        "656773377777600":["1989","BOV_1"],
        "077777774020731":["1990","H1"],
        "754175417672361":["1991","Manu1"],
        "677777477413000":["1992","EAI2-Manila"],
        "006377777760771":["1993","S"],
        "414343436742200":["1994","BOV"],
        "776377701760771":["1995","S"],
        "777400037760771":["1996","T1"],
        "554347436400261":["1997","AFRI"],
        "777777607740071":["1998","LAM9"],
        "777701607760771":["1999","LAM5"],
        "776377777760700":["2000","S"],
        "576337777760771":["2001","S"],
        "777417777700000":["2002","None"],
        "001677777760771":["2003","T1"],
        "776377777760071":["2004","S"],
        "017777600000371":["2005","None"],
        "400377607560771":["2006","LAM6"],
        "777737677760771":["2007","T3"],
        "577377774020771":["2008","H1"],
        "677177477413771":["2009","EAI2-Manila"],
        "400037777413741":["2010","EAI5"],
        "467777777413771":["2011","EAI5"],
        "677777761760771":["2012","T1"],
        "776177607760740":["2013","LAM3"],
        "776177607360771":["2014","LAM3"],
        "736177607760771":["2015","LAM3"],
        "737776777760601":["2016","X2"],
        "677777606060771":["2017","LAM11-ZWE"],
        "760760047700771":["2018","None"],
        "777776777760740":["2019","X1"],
        "700016777760771":["2020","X3"],
        "777777777700031":["2021","T"],
        "077776775760731":["2022","Ambiguous:T2 X1"],
        "777737777720171":["2023","H3"],
        "676363777777600":["2024","BOV"],
        "777777777700371":["2025","T"],
        "777777777740171":["2026","T1"],
        "777777777000731":["2027","None"],
        "777777604020771":["2028","None"],
        "777774376000711":["2029","None"],
        "777777704020751":["2030","H1"],
        "740000377760771":["2031","T1"],
        "037627777760771":["2032","T1"],
        "777777714160371":["2033","T1"],
        "577677777760771":["2034","T1"],
        "277777077413771":["2035","EAI2"],
        "777777774520771":["2036","H3"],
        "777777774360771":["2037","T1"],
        "603777607560771":["2038","LAM6"],
        "777776777560601":["2039","X2"],
        "777737777760371":["2040","T3"],
        "777777760000771":["2041","None"],
        "000000007660771":["2042","LAM3"],
        "777777737760740":["2043","T1"],
        "577777604020771":["2044","H1"],
        "776037607760771":["2045","LAM3"],
        "776377377760731":["2046","S"],
        "776163000000071":["2047","None"],
        "777737777320771":["2048","H3"],
        "777770777760771":["2049","T1"],
        "575777707760771":["2050","T1"],
        "777777760000140":["2051","None"],
        "737737777760771":["2052","T3"],
        "776377775760771":["2053","S"],
        "677777607760371":["2054","LAM"],
        "776377777720071":["2055","H3"],
        "770637777767671":["2056","AFRI_1"],
        "774767777420771":["2057","H3"],
        "774677607777071":["2058","AFRI"],
        "700000000000371":["2059","CAS"],
        "477017777413071":["2060","EAI3-IND"],
        "740177777013731":["2061","EAI1-SOM"],
        "777776777600000":["2062","None"],
        "700377707776671":["2063","AFRI"],
        "770757777767671":["2064","AFRI_1"],
        "776017407760771":["2065","LAM12-Madrid1"],
        "777777400060771":["2066","None"],
        "777771777760771":["2067","T1"],
        "377777763760771":["2068","T1"],
        "777777707760371":["2069","T1"],
        "777777637760771":["2070","T1"],
        "777720774760771":["2071","T3"],
        "000000004000171":["2072","None"],
        "776000607760771":["2073","LAM3"],
        "774177677413771":["2074","EAI5"],
        "740617777760771":["2075","T3-OSA"],
        "777776777420771":["2076","H3"],
        "757377777720771":["2077","H3"],
        "771347777740771":["2078","T1"],
        "777777776000740":["2079","None"],
        "777777776760571":["2080","T1"],
        "767777777720571":["2081","H3"],
        "577767777760731":["2082","T2"],
        "677777437413771":["2083","EAI2"],
        "677776477413771":["2084","EAI2-Manila"],
        "674377477413771":["2085","EAI2-Manila"],
        "747777770020771":["2086","H3"],
        "777737600020771":["2087","H3"],
        "777777737760731":["2088","T2"],
        "777773777720771":["2089","H3"],
        "777777761720771":["2090","H3"],
        "777637770020771":["2091","H3"],
        "777737770020771":["2092","H3"],
        "757777770020771":["2093","H3"],
        "777773700020731":["2094","H3"],
        "777037777760031":["2095","T1"],
        "077777477413771":["2096","EAI5"],
        "617777477413771":["2097","EAI5"],
        "647777477413731":["2098","EAI2-Manila"],
        "647777477413771":["2099","EAI2-Manila"],
        "700377700003771":["2100","CAS"],
        "000000000003000":["2101","Beijing"],
        "703760000000771":["2102","CAS"],
        "074000077777600":["2103","BOV"],
        "400017777413071":["2104","EAI3-IND"],
        "776176607760771":["2105","LAM3"],
        "777777400000371":["2106","None"],
        "776177400000131":["2107","None"],
        "100076777760771":["2108","X3"],
        "777775077560700":["2109","T1"],
        "777777607000771":["2110","None"],
        "701777777720771":["2111","H3"],
        "777717777760730":["2112","T2-Uganda"],
        "777177777760731":["2113","T2"],
        "777761777760771":["2114","T1"],
        "777777434760771":["2115","T1"],
        "777777774020701":["2116","H1"],
        "776377777763771":["2117","Manu2"],
        "777737347760411":["2118","Ambiguous:T4 T3"],
        "777600774413771":["2119","EAI4-VNM"],
        "007777707760351":["2120","T1"],
        "741737777760371":["2121","T3-OSA"],
        "741736777760771":["2122","T3-OSA"],
        "777577777777771":["2123","None"],
        "737777777777771":["2124","None"],
        "777777404760571":["2125","LAM7-TUR"],
        "743777700003371":["2126","EAI7-BGD2"],
        "777760007760751":["2127","T5-RUS1"],
        "777707777760771":["2128","T1"],
        "676773677377200":["2129","BOV_1"],
        "674773677777200":["2130","BOV"],
        "333737373737331":["2131","suppressed"],
        "776277777760771":["2132","S"],
        "676001677777200":["2133","BOV"],
        "740000007320771":["2134","H3"],
        "776367677760771":["2135","S"],
        "777777000320771":["2136","H3"],
        "177777774003731":["2137","EAI1-SOM"],
        "774777770020771":["2138","H3"],
        "740637777760771":["2139","T1"],
        "777777417760771":["2140","T1"],
        "636773677777600":["2141","BOV_1"],
        "677777777560771":["2142","T1"],
        "177777677760771":["2143","T1"],
        "773777776760731":["2144","T2"],
        "703777640003771":["2145","CAS1-Delhi"],
        "077577747003371":["2146","None"],
        "703777740003011":["2147","CAS1-Delhi"],
        "700000000000000":["2148","CAS"],
        "703757740000771":["2149","CAS"],
        "774077407777071":["2150","AFRI_3"],
        "777777347760771":["2151","T4"],
        "777777702003371":["2152","None"],
        "333373737373331":["2153","suppressed"],
        "777777434020771":["2154","H1"],
        "777777756013371":["2155","EAI5"],
        "777777757410000":["2156","EAI6-BGD1"],
        "770777774377671":["2157","AFRI_1"],
        "777777757410371":["2158","EAI6-BGD1"],
        "777777757411371":["2159","EAI6-BGD1"],
        "777737777700171":["2160","None"],
        "000000007740771":["2161","LAM3"],
        "777776004760771":["2162","LAM7-TUR"],
        "757377777700000":["2163","None"],
        "777777777740731":["2164","T2"],
        "777747607760771":["2165","LAM9"],
        "777776404360601":["2166","X2"],
        "703777740002331":["2167","CAS1-Delhi"],
        "557747606760771":["2168","LAM8"],
        "777776677760601":["2169","X2"],
        "777773777760571":["2170","T1"],
        "576777777720771":["2171","H3"],
        "370177607760771":["2172","LAM3"],
        "377777606060771":["2173","LAM11-ZWE"],
        "774037607777071":["2174","AFRI_2"],
        "760000003760771":["2175","T1"],
        "376147607760771":["2176","LAM3"],
        "774177406760771":["2177","LAM3"],
        "776376777760601":["2178","X2"],
        "776377760360771":["2179","S"],
        "777676777760601":["2180","X2"],
        "776371001760771":["2181","S"],
        "777377677763771":["2182","Manu2"],
        "764137607720771":["2183","H3"],
        "766177407760771":["2184","LAM3"],
        "774377003740771":["2185","T1"],
        "777016777760671":["2186","X1"],
        "777777347763771":["2187","Manu2"],
        "777377777760731":["2188","T2"],
        "777777761720740":["2189","H3"],
        "000177777420771":["2190","H4"],
        "777741007760771":["2191","LAM9"],
        "500076777560771":["2192","X3"],
        "777637763660771":["2193","T1"],
        "740377777760731":["2194","T2"],
        "477777376013771":["2195","EAI8-MDG"],
        "777775606060731":["2196","LAM11-ZWE"],
        "477776077413771":["2197","EAI5"],
        "417777777620771":["2198","H3"],
        "637737607760771":["2199","LAM2"],
        "777777607540761":["2200","LAM9"],
        "177777607760771":["2201","LAM9"],
        "777743557760771":["2202","T5-Madrid2"],
        "707777777720771":["2203","H3"],
        "760121400200771":["2204","None"],
        "000000007700371":["2205","None"],
        "676077477413771":["2206","EAI2-Manila"],
        "777377777413731":["2207","EAI1-SOM"],
        "761777777760771":["2208","T1"],
        "676600000000771":["2209","None"],
        "747773777460771":["2210","T1"],
        "757777707720671":["2211","H3"],
        "777777404720571":["2212","H3"],
        "775777777720751":["2213","H3"],
        "776777577720771":["2214","H3"],
        "767777770000771":["2215","None"],
        "777777147720771":["2216","H3"],
        "777777757760751":["2217","T5"],
        "777737677420771":["2218","H3"],
        "774003774020771":["2219","H1"],
        "000000007160771":["2220","LAM3"],
        "757737604020771":["2221","H1"],
        "770020607760771":["2222","LAM3"],
        "777000777760731":["2223","T2"],
        "776337777763771":["2224","Manu2"],
        "700076774360771":["2225","X3"],
        "777476777760771":["2226","X1"],
        "777777730020771":["2227","H3"],
        "776777757760771":["2228","T5"],
        "776177607763771":["2229","Manu2"],
        "777700077760771":["2230","None"],
        "700000004020711":["2231","H1"],
        "777737777700000":["2232","None"],
        "777763770000000":["2233","None"],
        "777737760000151":["2234","None"],
        "777777377740371":["2235","T4"],
        "477477777413071":["2236","EAI3-IND"],
        "702377740003771":["2237","CAS1-Delhi"],
        "471777777413071":["2238","EAI3-IND"],
        "741737777760671":["2239","T3-OSA"],
        "737777777000731":["2240","None"],
        "477776377413071":["2241","EAI3-IND"],
        "777777743440771":["2242","LAM10-CAM"],
        "774003777775071":["2243","AFRI_3"],
        "777577777760731":["2244","T2"],
        "774067600000071":["2245","AFRI"],
        "777760007760731":["2246","T5-RUS1"],
        "777740003763771":["2247","Manu2"],
        "777774607760771":["2248","LAM9"],
        "777600000060771":["2249","T1"],
        "737776777760731":["2250","Ambiguous:T2 X1"],
        "477740001760771":["2251","T1"],
        "777735777423771":["2252","Manu2"],
        "670177607760731":["2253","LAM3"],
        "730000777760771":["2254","T1-RUS2"],
        "707777770060731":["2255","T2"],
        "377700000020731":["2256","H3"],
        "777700004020771":["2257","H1"],
        "777777777700011":["2258","None"],
        "577637777420771":["2259","H4"],
        "777017577420771":["2260","H3"],
        "777737776000131":["2261","None"],
        "637760007760771":["2262","T5-RUS1"],
        "767777607760771":["2263","LAM9"],
        "777737607760711":["2264","Ambiguous:LAM5 LAM4"],
        "577777606060771":["2265","LAM11-ZWE"],
        "000037607760771":["2266","LAM3"],
        "701377400001771":["2267","CAS"],
        "777767606060771":["2268","LAM11-ZWE"],
        "703377400000771":["2269","CAS"],
        "742377777760771":["2270","S"],
        "677717607760771":["2271","LAM2"],
        "740777777760771":["2272","T1"],
        "777777607720731":["2273","H3"],
        "037777407560731":["2274","LAM"],
        "740000007760771":["2275","LAM3"],
        "777777777743771":["2276","None"],
        "703377600001771":["2277","CAS"],
        "777777606061771":["2278","None"],
        "777777777473771":["2279","Manu1"],
        "777727606060771":["2280","LAM11-ZWE"],
        "777775607760771":["2281","LAM9"],
        "776377777560771":["2282","S"],
        "777761377760731":["2283","Ambiguous:T4 T2"],
        "776177407760731":["2284","LAM3"],
        "777774777760771":["2285","T1"],
        "300076777760771":["2286","X3"],
        "000000007760711":["2287","LAM3"],
        "477777774000071":["2288","None"],
        "777717761720771":["2289","H3"],
        "777777607600171":["2290","None"],
        "777417777400000":["2291","None"],
        "677777607760071":["2292","LAM1"],
        "777737037760771":["2293","T3"],
        "777763774020771":["2294","H1"],
        "737776777560771":["2295","X1"],
        "737777777760601":["2296","T1"],
        "777763007760771":["2297","LAM9"],
        "777777647760771":["2298","T1"],
        "776107607760771":["2299","LAM3"],
        "743760047700771":["2300","None"],
        "776167607760671":["2301","LAM3"],
        "776117607760771":["2302","LAM3"],
        "743776777760601":["2303","X2"],
        "617776777760771":["2304","X1"],
        "600076777760771":["2305","X1"],
        "777737770000771":["2306","None"],
        "777703777560731":["2307","T2"],
        "777777757403771":["2308","None"],
        "774377003740371":["2309","T1"],
        "776177606760771":["2310","LAM"],
        "776177607760471":["2311","LAM3"],
        "776377777760611":["2312","S"],
        "776321777760771":["2313","S"],
        "737637400020771":["2314","H3"],
        "677777207760771":["2315","LAM1"],
        "000000000003740":["2316","Beijing"],
        "777377347760400":["2317","T4"],
        "674000000000071":["2318","AFRI"],
        "777770177413700":["2319","EAI5"],
        "777376777760771":["2320","X1"],
        "017777777320771":["2321","H3"],
        "777777760160731":["2322","T2"],
        "477774777413071":["2323","EAI3-IND"],
        "770737777777671":["2324","AFRI_1"],
        "777737777413701":["2325","EAI5"],
        "677777607760711":["2326","LAM"],
        "777776677760771":["2327","X1"],
        "777737757700011":["2328","None"],
        "777777776120771":["2329","H3"],
        "600077743760171":["2330","LAM10-CAM"],
        "777777605760471":["2331","LAM9"],
        "774037600760700":["2332","None"],
        "777777774020711":["2333","H1"],
        "777735777760730":["2334","T2-Uganda"],
        "000000777760730":["2335","T2-Uganda"],
        "777637607760771":["2336","LAM5"],
        "777377774020771":["2337","H1"],
        "777600777720771":["2338","H3"],
        "767777777760730":["2339","T2-Uganda"],
        "637777777760730":["2340","T2-Uganda"],
        "777733607760771":["2341","LAM5"],
        "037777607760771":["2342","LAM1"],
        "776177607400771":["2343","LAM2"],
        "776373001760771":["2344","S"],
        "000000007760760":["2345","LAM"],
        "777737776413731":["2346","EAI1-SOM"],
        "776000003660771":["2347","T1"],
        "577761777740371":["2348","T1"],
        "637777637760771":["2349","T1"],
        "600017607760771":["2350","LAM3"],
        "657777477413771":["2351","EAI2-Manila"],
        "747377777760771":["2352","T1"],
        "776177600000000":["2353","None"],
        "777737737420771":["2354","H3"],
        "777777345760471":["2355","T4-CEU1"],
        "717776777760771":["2356","X1"],
        "000000174020771":["2357","H1"],
        "777777777000071":["2358","None"],
        "703677740003171":["2359","CAS1-Delhi"],
        "700076777760271":["2360","X3"],
        "703776040000771":["2361","CAS"],
        "674000017000771":["2362","None"],
        "737700777413700":["2363","EAI5"],
        "703601740003771":["2364","CAS1-Delhi"],
        "777337577360771":["2365","T3"],
        "777777777760171":["2366","T1"],
        "766377777760771":["2367","S"],
        "777737206060771":["2368","LAM11-ZWE"],
        "777776777560731":["2369","Ambiguous:T2 X1"],
        "276773777777600":["2370","BOV_1"],
        "400777777620731":["2371","H3"],
        "600777607760771":["2372","LAM1"],
        "700337740003771":["2373","CAS2"],
        "717737777760771":["2374","T3"],
        "767777774020771":["2375","H1"],
        "770000000020771":["2376","H3"],
        "770777777420711":["2377","H3"],
        "776162000000071":["2378","None"],
        "777600000000000":["2379","None"],
        "777623405760471":["2380","LAM5"],
        "777677777720771":["2381","H3"],
        "777723400000360":["2382","None"],
        "333777333777331":["2383","SIT missing"],
        "777740003760731":["2384","T2"],
        "777777606060471":["2385","LAM11-ZWE"],
        "777777607400171":["2386","None"],
        "777777774000331":["2387","None"],
        "746177607760771":["2388","LAM3"],
        "000202000002771":["2389","None"],
        "577737747420771":["2390","H4"],
        "703377400003771":["2391","CAS1-Kili"],
        "703777700003171":["2392","CAS"],
        "777037777760771":["2393","T1"],
        "767700777760731":["2394","T2"],
        "777777767413771":["2395","EAI5"],
        "377776777760711":["2396","X1"],
        "700076377760771":["2397","T1"],
        "703777744037771":["2398","None"],
        "700036777760671":["2399","X3"],
        "437777774020771":["2400","H1"],
        "777777607600371":["2401","None"],
        "776177607760711":["2402","LAM3"],
        "677737601760771":["2403","LAM"],
        "577777750000000":["2404","None"],
        "600377477413771":["2405","EAI5"],
        "777777737760471":["2406","T1"],
        "770003400003071":["2407","AFRI_2"],
        "777774000000771":["2408","None"],
        "777737757760771":["2409","Ambiguous:T3 T5"],
        "376177607760731":["2410","LAM3"],
        "700076777760711":["2411","X3"],
        "777737607703771":["2412","None"],
        "000000000003760":["2413","Beijing"],
        "777737674020771":["2414","H1"],
        "776177607700771":["2415","None"],
        "770777407560731":["2416","LAM6"],
        "777777544320771":["2417","H3"],
        "767777767760771":["2418","T1"],
        "703000000000371":["2419","CAS"],
        "777777777720700":["2420","H3"],
        "777437737700000":["2421","None"],
        "774037777360700":["2422","T1"],
        "777777437760731":["2423","T2"],
        "777775777760700":["2424","T1"],
        "772174607760771":["2425","LAM3"],
        "777777777000271":["2426","None"],
        "377777777000031":["2427","None"],
        "674017607760771":["2428","LAM3"],
        "777777743560771":["2429","LAM10-CAM"],
        "333555333355331":["2430","lacks binary and octal spoligotype"],
        "677777607760671":["2431","LAM1"],
        "333444455533331":["2432","suppressed"],
        "333444555222331":["2433","suppressed"],
        "333444555223331":["2434","suppressed"],
        "777777777677731":["2435","None"],
        "776177607020771":["2436","H3"],
        "777077777760771":["2437","T1"],
        "776777606020771":["2438","H3"],
        "776171000000071":["2439","None"],
        "770777777740671":["2440","AFRI_1"],
        "770777777600671":["2441","AFRI_1"],
        "770773777777671":["2442","AFRI_1"],
        "760000377777671":["2443","AFRI"],
        "740377777777671":["2444","AFRI"],
        "477777770000000":["2445","None"],
        "060000777777671":["2446","AFRI"],
        "777777767760760":["2447","T1"],
        "774337607760700":["2448","Ambiguous:LAM5 LAM4"],
        "747777600000000":["2449","None"],
        "777777774412071":["2450","EAI4-VNM"],
        "577777777413711":["2451","EAI5"],
        "477774377413071":["2452","EAI3-IND"],
        "777776775413700":["2453","EAI5"],
        "414367777413771":["2454","EAI5"],
        "777716777760601":["2455","X2"],
        "646573777777600":["2456","BOV_1"],
        "407777777413071":["2457","EAI3-IND"],
        "776036777760771":["2458","X1"],
        "476000377413071":["2459","EAI3-IND"],
        "777774777760601":["2460","T1"],
        "500076000000000":["2461","CAS"],
        "613776777760601":["2462","X2"],
        "777037777413731":["2463","EAI1-SOM"],
        "700036777760730":["2464","X3"],
        "477777757413771":["2465","EAI6-BGD1"],
        "000036777760771":["2466","X1"],
        "477777775413760":["2467","EAI5"],
        "777777757760711":["2468","T5"],
        "777773676760730":["2469","T2-Uganda"],
        "477777077413771":["2470","EAI5"],
        "600000177760711":["2471","T1"],
        "037777777000371":["2472","None"],
        "037775770060771":["2473","T1"],
        "777740777760531":["2474","T2"],
        "676753777677600":["2475","BOV_1"],
        "700076777760600":["2476","X3"],
        "617777774020771":["2477","H1"],
        "776177730000000":["2478","None"],
        "700036776060771":["2479","X3"],
        "777767777413600":["2480","EAI5"],
        "776267777760771":["2481","S"],
        "776377777413731":["2482","EAI1-SOM"],
        "777777606060770":["2483","LAM11-ZWE"],
        "737377677760771":["2484","T1"],
        "603777740003171":["2485","CAS1-Delhi"],
        "701577600023171":["2486","Manu2"],
        "777777606060751":["2487","LAM11-ZWE"],
        "741777606060771":["2488","LAM11-ZWE"],
        "777707606060771":["2489","LAM11-ZWE"],
        "776007606060731":["2490","LAM11-ZWE"],
        "777777204060731":["2491","LAM4"],
        "777677606060771":["2492","LAM11-ZWE"],
        "777773606060731":["2493","LAM11-ZWE"],
        "377737606060771":["2494","LAM11-ZWE"],
        "777777606000771":["2495","None"],
        "777367777760771":["2496","T1"],
        "677677607760771":["2497","LAM1"],
        "777760077620771":["2498","H3"],
        "777777617760771":["2499","T1"],
        "776377770000771":["2500","None"],
        "777777774220771":["2501","H3"],
        "777777407560771":["2502","LAM6"],
        "777777777420611":["2503","H3"],
        "333465465653331":["2504","SIT missing"],
        "777677607560771":["2505","LAM6"],
        "677757607760771":["2506","LAM1"],
        "777777400600000":["2507","None"],
        "777577600000000":["2508","None"],
        "777777777750371":["2509","None"],
        "774737777760731":["2510","Ambiguous:T3 T2"],
        "777736770000000":["2511","None"],
        "657757607760771":["2512","LAM1"],
        "777767777760471":["2513","T1"],
        "774377607760700":["2514","LAM4"],
        "777737607760400":["2515","Ambiguous:LAM5 LAM4"],
        "756377777760771":["2516","S"],
        "777600167760771":["2517","T1"],
        "763763607560771":["2518","LAM6"],
        "700000007560771":["2519","LAM3"],
        "771600000000011":["2520","None"],
        "737777607560771":["2521","LAM6"],
        "677777607760740":["2522","LAM1"],
        "777740017160771":["2523","T1"],
        "757777605560771":["2524","LAM6"],
        "677737607620771":["2525","None"],
        "007777606000031":["2526","None"],
        "776177607660771":["2527","LAM3"],
        "374377607760700":["2528","LAM4"],
        "777777607740371":["2529","LAM9"],
        "777700017760771":["2530","T1"],
        "677777777760601":["2531","T1"],
        "777777634020771":["2532","H1"],
        "777760007560771":["2533","T5-RUS1"],
        "777377607700771":["2534","None"],
        "777777207600331":["2535","None"],
        "676777607760771":["2536","LAM1"],
        "777617777720771":["2537","H3"],
        "333456654333331":["2538","SIT missing"],
        "673777607760771":["2539","LAM1"],
        "777700000760771":["2540","T1"],
        "777740000760771":["2541","T1"],
        "777400167760771":["2542","T1"],
        "773601757013371":["2543","EAI5"],
        "677737607760371":["2544","LAM2"],
        "703660000000331":["2545","CAS"],
        "777600073760771":["2546","T1"],
        "777740000660771":["2547","T1"],
        "777777607600331":["2548","None"],
        "777677207760771":["2549","LAM9"],
        "777737743760771":["2550","LAM10-CAM"],
        "777777607760701":["2551","LAM4"],
        "677637607760771":["2552","LAM2"],
        "776776777760601":["2553","X2"],
        "577777207760771":["2554","LAM9"],
        "776777777760601":["2555","T1"],
        "777776607760601":["2556","X2"],
        "777777777540771":["2557","T1"],
        "333555666777331":["2558","SIT missing"],
        "333222333414331":["2559","SIT missing"],
        "333111222444331":["2560","SIT missing"],
        "333666555424331":["2561","SIT missing"],
        "777377600000000":["2562","None"],
        "777777607700731":["2563","LAM4"],
        "377777600000000":["2564","None"],
        "377776777760601":["2565","X2"],
        "777777607160771":["2566","LAM9"],
        "333765567456331":["2567","SIT missing"],
        "737776607760771":["2568","LAM9"],
        "747777667560771":["2569","T1"],
        "777777337760771":["2570","T4"],
        "776340000000171":["2571","None"],
        "667777607760771":["2572","LAM1"],
        "777777600000731":["2573","None"],
        "777776777760600":["2574","X2"],
        "774777747420731":["2575","H3"],
        "777723607760771":["2576","LAM5"],
        "777477777760771":["2577","T1"],
        "777737777760740":["2578","T3"],
        "776000000760771":["2579","T1"],
        "777740016360771":["2580","T1"],
        "770023206371071":["2581","AFRI"],
        "737777377760731":["2582","Ambiguous:T4 T2"],
        "477777277413571":["2583","EAI8-MDG"],
        "777770677760771":["2584","T1"],
        "703763740003171":["2585","CAS1-Delhi"],
        "077577717003351":["2586","None"],
        "777737777777731":["2587","None"],
        "676023477413771":["2588","EAI2-Manila"],
        "477677376003741":["2589","None"],
        "677777470000071":["2590","EAI2-Manila"],
        "703767740000000":["2591","CAS"],
        "677377477413771":["2592","EAI2-Manila"],
        "400037777413761":["2593","EAI5"],
        "767777777413731":["2594","EAI1-SOM"],
        "757777606760631":["2595","LAM8"],
        "777737777760760":["2596","T3"],
        "777777777730771":["2597","Manu3"],
        "777777557770771":["2598","Manu3"],
        "677777744030771":["2599","Manu3"],
        "776177607770771":["2600","Manu3"],
        "577777777770771":["2601","Manu3"],
        "777777607770771":["2602","Manu3"],
        "776177607770731":["2603","Manu3"],
        "677777607770771":["2604","Manu3"],
        "777777557010771":["2605","EAI5"],
        "776017607770771":["2606","Manu3"],
        "776177400010171":["2607","EAI5"],
        "477777777730771":["2608","Manu3"],
        "777777607770770":["2609","Manu3"],
        "000000000003770":["2610","Beijing"],
        "757777777030771":["2611","Manu3"],
        "757777774030771":["2612","Manu3"],
        "777777777400001":["2613","None"],
        "777367777720671":["2614","H3"],
        "777367777720771":["2615","H3"],
        "776167777760771":["2616","T1"],
        "777367577760771":["2617","T1"],
        "777367777760671":["2618","T1"],
        "777367777720661":["2619","H3"],
        "777367607760661":["2620","LAM9"],
        "333543345765331":["2621","SIT missing"],
        "777377777760671":["2622","T1"],
        "700000004020771":["2623","H1"],
        "400000007760771":["2624","LAM3"],
        "776163400000171":["2625","None"],
        "776177607700171":["2626","LAM3"],
        "773767777720771":["2627","H3"],
        "777377777520771":["2628","H3"],
        "774377777420771":["2629","H3"],
        "676773477777600":["2630","BOV_1"],
        "676773675777600":["2631","BOV_1"],
        "777777607420771":["2632","H3"],
        "000007777760771":["2633","T1"],
        "777577777620771":["2634","H3"],
        "777767777000271":["2635","None"],
        "777763777777600":["2636","None"],
        "776737777420771":["2637","H3"],
        "777737607400071":["2638","None"],
        "776377737560731":["2639","S"],
        "677777777720731":["2640","H3"],
        "776376037760751":["2641","S"],
        "777777774020611":["2642","H1"],
        "760177777720771":["2643","H3"],
        "177777777720771":["2644","H3"],
        "777777662020771":["2645","H3"],
        "766177607760771":["2646","LAM3"],
        "777077607760771":["2647","LAM9"],
        "757767607760671":["2648","LAM9"],
        "777777607400371":["2649","None"],
        "743737607760731":["2650","Ambiguous:LAM5 LAM4"],
        "777737604760771":["2651","LAM5"],
        "777777604760771":["2652","LAM9"],
        "776377707760771":["2653","S"],
        "777637607400031":["2654","LAM"],
        "775777760060771":["2655","T1"],
        "477377777413071":["2656","EAI3-IND"],
        "703777700002771":["2657","CAS"],
        "777700000001331":["2658","None"],
        "740027606760771":["2659","LAM3"],
        "777777777600131":["2660","None"],
        "777677577413771":["2661","EAI5"],
        "776773777777600":["2662","None"],
        "777076777760771":["2663","X1"],
        "776377770000000":["2664","None"],
        "777777777000771":["2665","None"],
        "367777770020731":["2666","H3"],
        "333234245356331":["2667","SIT missing"],
        "777777377760770":["2668","T4"],
        "000000000000000":["2669","3 or more families have deleted DR"],
        "703607740003171":["2670","CAS1-Delhi"],
        "467777377413771":["2671","EAI8-MDG"],
        "777777377413771":["2672","EAI5"],
        "777677777760731":["2673","T2"],
        "777757777760731":["2674","T2"],
        "477777777413171":["2675","EAI5"],
        "777774377413671":["2676","EAI5"],
        "403637757413771":["2677","EAI6-BGD1"],
        "777777776023771":["2678","Manu2"],
        "477775777760771":["2679","T1"],
        "703740000003771":["2680","CAS"],
        "477777777413761":["2681","EAI5"],
        "703761000000331":["2682","CAS"],
        "703777600003171":["2683","CAS1-Delhi"],
        "703770000000001":["2684","CAS"],
        "703777000003171":["2685","CAS"],
        "703767740003471":["2686","CAS1-Delhi"],
        "577777757473771":["2687","Manu1"],
        "703777742003571":["2688","None"],
        "777775000000171":["2689","None"],
        "703777740000071":["2690","CAS"],
        "037737777760771":["2691","T3"],
        "777737707760771":["2692","T3"],
        "703763740003771":["2693","CAS1-Delhi"],
        "303777740003171":["2694","CAS1-Delhi"],
        "703767740003571":["2695","CAS1-Delhi"],
        "703777760003771":["2696","CAS1-Delhi"],
        "775777777763771":["2697","Manu2"],
        "777777743763771":["2698","Manu2"],
        "777777747760731":["2699","T2"],
        "477777770020771":["2700","H3"],
        "777777277413771":["2701","EAI5"],
        "676143777777600":["2702","BOV"],
        "400027777413471":["2703","EAI5"],
        "677777457413771":["2704","EAI2-Manila"],
        "777403607760771":["2705","LAM5"],
        "701767777700011":["2706","None"],
        "701767740003671":["2707","CAS"],
        "701377740003771":["2708","CAS"],
        "773777607560171":["2709","LAM6"],
        "577777777411731":["2710","EAI1-SOM"],
        "477777740173071":["2711","Manu1"],
        "037637777770171":["2712","None"],
        "700277740007001":["2713","None"],
        "703777774000771":["2714","None"],
        "477777760177071":["2715","None"],
        "777777637560771":["2716","T1"],
        "573377777420771":["2717","H4"],
        "777477777460731":["2718","T2"],
        "377777047341771":["2719","None"],
        "677777477413071":["2720","EAI2-Manila"],
        "777777700000171":["2721","None"],
        "777737774413731":["2722","EAI4-VNM"],
        "577777777760371":["2723","T1"],
        "774737777420771":["2724","H3"],
        "777777606160771":["2725","LAM9"],
        "703777600000771":["2726","CAS1-Delhi"],
        "414363777777600":["2727","BOV"],
        "577767777420771":["2728","H4"],
        "614363777777600":["2729","BOV"],
        "637774777740130":["2730","T2-Uganda"],
        "703777740001071":["2731","CAS"],
        "777730607760771":["2732","LAM5"],
        "777777760000371":["2733","None"],
        "777777601560751":["2734","T1"],
        "333111222333331":["2735","SIT missing"],
        "333222333444331":["2736","SIT missing"],
        "333333444555331":["2737","SIT missing"],
        "333551444333331":["2738","SIT missing"],
        "333662555444331":["2739","SIT missing"],
        "777377757760771":["2740","T5"],
        "333132132132331":["2741","SIT missing"],
        "333142142142331":["2742","SIT missing"],
        "333152152152331":["2743","SIT missing"],
        "333162162162331":["2744","SIT missing"],
        "333172172172331":["2745","SIT missing"],
        "333243243243331":["2746","SIT missing"],
        "333253253253331":["2747","SIT missing"],
        "333263263263331":["2748","SIT missing"],
        "333273273273331":["2749","SIT missing"],
        "333465665465331":["2750","SIT missing"],
        "333475775475331":["2751","SIT missing"],
        "333576176576331":["2752","SIT missing"],
        "333675675675331":["2753","SIT missing"],
        "333777666555331":["2754","SIT missing"],
        "333666555444331":["2755","SIT missing"],
        "333555444333331":["2756","SIT missing"],
        "333444333222331":["2757","SIT missing"],
        "333333222111331":["2758","SIT missing"],
        "333212212212331":["2759","SIT missing"],
        "333312312312331":["2760","SIT missing"],
        "333412412412331":["2761","SIT missing"],
        "333512512512331":["2762","SIT missing"],
        "333612612612331":["2763","SIT missing"],
        "333712712712331":["2764","SIT missing"],
        "333323323323331":["2765","SIT missing"],
        "333423423423331":["2766","SIT missing"],
        "333523523523331":["2767","SIT missing"],
        "333623623623331":["2768","SIT missing"],
        "333723723723331":["2769","SIT missing"],
        "333431431431331":["2770","SIT missing"],
        "333531531531331":["2771","SIT missing"],
        "333631631631331":["2772","SIT missing"],
        "333731731731331":["2773","SIT missing"],
        "333476476476331":["2774","SIT missing"],
        "777777757767771":["2775","None"],
        "333576576576331":["2776","SIT missing"],
        "333564564564331":["2777","SIT missing"],
        "333543543543331":["2778","SIT missing"],
        "333475475475331":["2779","SIT missing"],
        "333465465465331":["2780","SIT missing"],
        "777777777767771":["2781","None"],
        "777777600003271":["2782","None"],
        "703777740003751":["2783","CAS1-Delhi"],
        "757777607460731":["2784","LAM4"],
        "777347777760371":["2785","T1"],
        "777777775760770":["2786","T1"],
        "333717777177331":["2787","SIT missing"],
        "333727777277331":["2788","SIT missing"],
        "333737777377331":["2789","SIT missing"],
        "333747777477331":["2790","SIT missing"],
        "333757777577331":["2791","SIT missing"],
        "333767777777331":["2792","SIT missing"],
        "333777777677331":["2793","SIT missing"],
        "333712777777331":["2794","SIT missing"],
        "333713777777331":["2795","SIT missing"],
        "333714777777331":["2796","SIT missing"],
        "333715777777331":["2797","SIT missing"],
        "333716777777331":["2798","SIT missing"],
        "333717777777331":["2799","SIT missing"],
        "333721777777331":["2800","SIT missing"],
        "333722777777331":["2801","SIT missing"],
        "333723777777331":["2802","SIT missing"],
        "333724777777331":["2803","SIT missing"],
        "333725777777331":["2804","SIT missing"],
        "333726777777331":["2805","SIT missing"],
        "333727777777331":["2806","SIT missing"],
        "333731777777331":["2807","SIT missing"],
        "333732777777331":["2808","SIT missing"],
        "333733777777331":["2809","SIT missing"],
        "333734777777331":["2810","SIT missing"],
        "333735777777331":["2811","SIT missing"],
        "333736777777331":["2812","SIT missing"],
        "333737777777331":["2813","SIT missing"],
        "333741777777331":["2814","SIT missing"],
        "333742777777331":["2815","SIT missing"],
        "333743777777331":["2816","SIT missing"],
        "333744777777331":["2817","SIT missing"],
        "333745777777331":["2818","SIT missing"],
        "333746777777331":["2819","SIT missing"],
        "333747777777331":["2820","SIT missing"],
        "333751777777331":["2821","SIT missing"],
        "333752777777331":["2822","SIT missing"],
        "333753777777331":["2823","SIT missing"],
        "333754777777331":["2824","SIT missing"],
        "333755777777331":["2825","SIT missing"],
        "333756777777331":["2826","SIT missing"],
        "333757771277331":["2827","SIT missing"],
        "333761777777331":["2828","SIT missing"],
        "333762777777331":["2829","SIT missing"],
        "333763777777331":["2830","SIT missing"],
        "333764777777331":["2831","SIT missing"],
        "333765777777331":["2832","SIT missing"],
        "333766777777331":["2833","SIT missing"],
        "333757777777331":["2834","SIT missing"],
        "333771777777331":["2835","SIT missing"],
        "333772777777331":["2836","SIT missing"],
        "333773777777331":["2837","SIT missing"],
        "333774777777331":["2838","SIT missing"],
        "333775777777331":["2839","SIT missing"],
        "333776777777331":["2840","SIT missing"],
        "333777777777331":["2841","SIT missing"],
        "333771277777331":["2842","SIT missing"],
        "333772277777331":["2843","SIT missing"],
        "333773277777331":["2844","SIT missing"],
        "333774277777331":["2845","SIT missing"],
        "333775277777331":["2846","SIT missing"],
        "333776277777331":["2847","SIT missing"],
        "333777277777331":["2848","SIT missing"],
        "333771217777331":["2849","SIT missing"],
        "333772217777331":["2850","SIT missing"],
        "333773217777331":["2851","SIT missing"],
        "333774217777331":["2852","SIT missing"],
        "333775217777331":["2853","SIT missing"],
        "333776217777331":["2854","SIT missing"],
        "333777237777331":["2855","SIT missing"],
        "333777247777331":["2856","SIT missing"],
        "333777257777331":["2857","SIT missing"],
        "333777267777331":["2858","SIT missing"],
        "3337772727777331":["2859","SIT missing"],
        "333777227777331":["2860","SIT missing"],
        "777765757413371":["2861","EAI6-BGD1"],
        "774777777413771":["2862","EAI5"],
        "657777777413071":["2863","EAI5"],
        "700777747413671":["2864","EAI6-BGD1"],
        "777777607460771":["2865","LAM9"],
        "777717774020771":["2866","H1"],
        "777777775760731":["2867","T2"],
        "777777363760771":["2868","T4"],
        "377777364020771":["2869","H1"],
        "377777767760771":["2870","T1"],
        "777037777760611":["2871","T1"],
        "777777557760700":["2872","T5-Madrid2"],
        "676773777700200":["2873","BOV_1"],
        "264073737777600":["2874","BOV_2"],
        "774067607761031":["2875","None"],
        "774067606767031":["2876","None"],
        "574063670007071":["2877","AFRI_3"],
        "074000430000600":["2878","BOV_4-Caprae"],
        "074000437777600":["2879","BOV"],
        "772763000001671":["2880","None"],
        "772763400001671":["2881","None"]}


In [ ]:
# 7-UPDATED) Dictionary is already loaded in cell 7 – no external file needed
print(f"✅ Dictionary already loaded: {len(sit_fam):,} entries available for SIT/family lookup.")


✅ Dictionary already loaded: 2,881 entries available for SIT/family lookup.


In [ ]:
# 8bis) Normalise + SANITISE the SITVIT mapping
#   - coerce every value to (SIT, family) as strings
#   - keep only valid 15-digit octal keys (digits 0-7); drop & log the rest
#   - this removes known malformed entries (e.g. a non-numeric key and a 16-digit key)
clean, dropped, conflicts = {}, [], []
for k, v in sit_fam.items():
    if isinstance(v, (list, tuple)):
        sit = str(v[0]) if len(v) >= 1 else "NA"
        fam = str(v[1]) if len(v) >= 2 else "NA"
    else:
        sit, fam = str(v), "NA"
    ks = str(k).strip()
    if not ks.isdigit():
        dropped.append((k, v)); continue
    ks = ks.zfill(15)
    if len(ks) != 15 or any(c > "7" for c in ks):   # not a valid 15-digit octal code
        dropped.append((k, v)); continue
    if ks in clean and clean[ks] != (sit, fam):
        conflicts.append((ks, clean[ks], (sit, fam)))
    clean[ks] = (sit, fam)

sit_fam = clean
print(f"✅ SITVIT dictionary sanitised: {len(sit_fam):,} valid 15-digit entries.")
if dropped:
    print(f"   ⚠️ dropped {len(dropped)} malformed entr(y/ies): "
          f"{[d[0] for d in dropped][:5]}{'...' if len(dropped) > 5 else ''}")
if conflicts:
    print(f"   ⚠️ {len(conflicts)} octal code(s) had conflicting SIT/family (kept last), "
          f"e.g. {conflicts[:2]}")


✅ SITVIT dictionary sanitised: 2,879 valid 15-digit entries.


In [ ]:
# 8) Core processing — assay-aware, fast scanner, per-sample error isolation
from tqdm import tqdm
import statistics

# ---------- 8.1 fast spacer scanner (Aho-Corasick, else pure-Python k-mer set) ----------
SPACER_LEN = len(fsp[0])
if not all(len(s) == SPACER_LEN for s in fsp + crsp):
    raise ValueError("All spacer k-mers must be the same length for the k-mer scanner.")

try:
    import ahocorasick
    _A = ahocorasick.Automaton()
    for _i, _s in enumerate(fsp):  _A.add_word(_s, _i)   # forward orientation
    for _i, _s in enumerate(crsp): _A.add_word(_s, _i)   # reverse complement -> same index
    _A.make_automaton()
    def spacer_hits(seq):
        return {idx for _, idx in _A.iter(seq)}
    SCANNER = "aho-corasick"
except Exception:
    _KMAP = {}
    for _i, _s in enumerate(fsp):  _KMAP.setdefault(_s, _i)
    for _i, _s in enumerate(crsp): _KMAP.setdefault(_s, _i)
    _K = SPACER_LEN
    def spacer_hits(seq):
        km, hits = _KMAP, set()
        for j in range(len(seq) - _K + 1):
            idx = km.get(seq[j:j + _K])
            if idx is not None:
                hits.add(idx)
        return hits
    SCANNER = "kmer-set (pure python)"
print(f"Spacer scanner: {SCANNER}")

# ---------- 8.2 tolerant FASTQ reading ----------
def open_text(p):
    return gzip.open(p, "rt") if str(p).endswith(".gz") else open(p, "r")

def iter_read_seqs(path):
    """Yield UPPERCASE sequence lines. Reads strict 4-line records and stops
    cleanly on a truncated final record instead of raising."""
    with open_text(path) as fh:
        while True:
            h = fh.readline()
            if not h:
                break
            seq = fh.readline()
            fh.readline()            # '+'
            q = fh.readline()        # quality
            if not q:
                break                # truncated record -> stop
            yield seq.strip().upper()

# ---------- 8.3 scan one sample ----------
def scan_sample(files):
    cnt = [0] * 43
    reads = 0
    reads_with_hit = 0
    for path in files:
        for seq in iter_read_seqs(path):
            reads += 1
            hits = spacer_hits(seq)
            if hits:
                reads_with_hit += 1
                for idx in hits:
                    cnt[idx] += 1
    return cnt, reads, reads_with_hit

# ---------- 8.4 assay-aware presence calling ----------
def resolve_assay(hit_frac):
    if assay in ("HiSpoT", "WGS"):
        return assay
    return "HiSpoT" if hit_frac >= auto_enrich_frac else "WGS"   # auto

def call_pattern(cnt, used_assay):
    total = sum(cnt)
    mean = total / 43.0                       # legacy: average over ALL 43 spacers
    if used_assay == "HiSpoT":
        cut = max(hispot_min_reads, mean * threshold)
        est = round(mean, 2)
    else:                                     # WGS: robust, coverage-aware
        nz = sorted(c for c in cnt if c > 0)
        est = nz[len(nz) // 2] if nz else 0   # median of non-zero spacer depths
        cut = max(wgs_min_reads, est * wgs_frac)
    binv = [1 if c >= cut else 0 for c in cnt]
    return binv, mean, cut, est

def qc_verdict(used_assay, reads, n_present, mean, est):
    if reads == 0:
        return "EMPTY_FILE"
    if n_present == 0:
        return "NO_SPACERS"
    if used_assay == "HiSpoT" and mean < hispot_qc_min_mean:
        return "LOW_COVERAGE"
    if used_assay == "WGS" and est < wgs_qc_min_depth:
        return "LOW_COVERAGE"
    return "PASS"

# ---------- 8.5 octal / binary helpers ----------
def bin_to_octal(b):
    return "".join(str(b[i] * 4 + b[i + 1] * 2 + b[i + 2]) for i in range(0, 42, 3)) + str(b[42])
def bin_to_str(b):
    return "".join(map(str, b))

# ---------- 8.6 run all samples (errors isolated per sample) ----------
rows, errors = [], []
for name, files in tqdm(samples, desc="Processing"):
    try:
        cnt, reads, reads_with_hit = scan_sample(files)
        hit_frac = (reads_with_hit / reads) if reads else 0.0
        used_assay = resolve_assay(hit_frac)
        binv, mean, cut, est = call_pattern(cnt, used_assay)
        octal = bin_to_octal(binv)
        sit, fam = sit_fam.get(str(octal).zfill(15), ("NA", "NA"))
        n_present = sum(binv)
        rows.append(dict(
            sample=name,
            binary=bin_to_str(binv),
            octal=octal,
            SIT=sit,
            family=fam,
            total_reads=reads,
            mean_per_spacer=round(mean, 2),
            cutoff=round(cut, 2),
            assay=used_assay,
            n_present=n_present,
            est_depth=est,
            hit_frac=round(hit_frac, 4),
            qc=qc_verdict(used_assay, reads, n_present, mean, est),
        ))
    except Exception as e:
        errors.append((name, repr(e)))
        rows.append(dict(
            sample=name, binary="", octal="", SIT="NA", family="NA",
            total_reads=0, mean_per_spacer=0, cutoff=0, assay="NA",
            n_present=0, est_depth=0, hit_frac=0, qc="ERROR",
        ))

summary = pd.DataFrame(rows, columns=[
    "sample", "binary", "octal", "SIT", "family",
    "total_reads", "mean_per_spacer", "cutoff",
    "assay", "n_present", "est_depth", "hit_frac", "qc",
])

if errors:
    print(f"⚠️ {len(errors)} sample(s) failed and are marked qc=ERROR:")
    for n, e in errors[:10]:
        print(f"   - {n}: {e}")

n_pass = int((summary["qc"] == "PASS").sum())
print(f"Done. {len(summary)} sample(s); {n_pass} PASS, {len(summary) - n_pass} flagged.")
summary


Spacer scanner: aho-corasick


Processing: 100%|██████████| 1/1 [01:37<00:00, 97.67s/it]

Done. 1 sample(s); 1 PASS, 0 flagged.


,sample,binary,octal,SIT,family,total_reads,mean_per_spacer,cutoff,assay,n_present,est_depth,hit_frac,qc
0,T3_0525-11_S7_L001,1111111110000000000111111111111100001111111,777000377760771,149,T3-ETH,4605400,137.3,19.0,WGS,29,190,0.0006,PASS


In [ ]:
# 9) Save Excel & TXT
excel_path = os.path.join(output_folder, "clinical_spoligotyping_results.xlsx")
txt_path   = os.path.join(output_folder, "clinical_spoligotyping_results.txt")

try:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as xw:
        summary.to_excel(xw, index=False, sheet_name="results")
    summary.to_csv(txt_path, sep="\t", index=False)
    print("Saved to", output_folder)
except Exception as e:
    print("⚠️ Could not write to the output folder:", repr(e))
    print("   Check that Google Drive is mounted and has free space.")


Saved to /content/drive/MyDrive/HiSpoT_Fastq/20260611_115140


In [ ]:
# 10) Optional download
from google.colab import files
import glob
for f in glob.glob(os.path.join(output_folder, "clinical_spoligotyping_results.*")):
    try:
        files.download(f)
    except Exception:
        pass


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>